In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import joblib
import pymssql
import calendar
import datetime
import sqlalchemy as db
import pickle
import warnings
import sys
from copy import deepcopy, copy
sys.path.append('path/miniconda3/lib/python3.12/site-packages')
import xgboost

from email import encoders
from email.mime.application import MIMEApplication
from email.mime.multipart import MIMEMultipart, MIMEBase
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.utils import COMMASPACE, formatdate
from jinja2 import Environment
import smtplib


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
def db_connection():
    con = pymssql.connect(user='',password = '',host='',database='',appname='',autocommit = True)
    cursor = con.cursor()
    return con, cursor 
def disconnect(con, cursor):
    con.close()
    cursor.close() 

In [3]:
def fetch_data_from_sql(start_date, end_date):
    query = f"""query"""
    conn, cursor = connection_76()
    df = pd.read_sql(query, conn)
    disconnect(conn, cursor)
    return df

In [4]:
def provider_payer():
        output_2 = []
        query = f"""
        query
        """

        con, cursor = connection_76()

        df_res = pd.read_sql(query, con)
        output_2.append(df_res)
        result_df = pd.concat(output_2, ignore_index=True)
        disconnect(con, cursor)
        
        return result_df

In [5]:
def all_submitted_claims(start_date, end_date):
        output_2 = []
        query = f"""
            query
        """

        con, cursor = connection_76()

        df_res = pd.read_sql(query, con)
        output_2.append(df_res)
        result_df = pd.concat(output_2, ignore_index=True)
        disconnect(con, cursor)
        
        return result_df

In [6]:
def preprocess_data(df):
    df['Process_Date'] = pd.to_datetime(df['Process_Date'])
    df['Day_of_Month'] = df['Process_Date'].dt.day
    df['Weekday'] = df['Process_Date'].dt.weekday
    df['Weekday_Name'] = df['Process_Date'].dt.day_name()
    df=df[['Claim_No','Billing_Physician','Amt_Due','Claim_Total','Pri_Status','Sec_Status','Oth_Status','Pat_Status',
    'Submission_Type', 'STATE', 'POS', 'InsPayer_Id','Insurance_State','PLAN_NAME_TYPE','DX_Code1','DX_Code2',
    'DX_Code3','PRIMARY_CARE_PHYSICIAN', 'Provider_Code','Practice_Code', 'TAXONOMY_CODE', 'Procedure_Code',
    'DX_Pointer1','InsName_Id', 'DX_Pointer2', 'DX_Pointer3', 'DX_Pointer4', 'Modi_Code1', 'Modi_Code2',
    'Modi_Code3', 'Modi_Code4', 'Amount', 'units', 'Day_of_Month', 'Weekday', 'Process_Date']]
    df['Modi_Code1'].fillna('', inplace=True)
    df['Modi_Code2'].fillna('', inplace=True)
    df['Modi_Code3'].fillna('', inplace=True)
    df['Modi_Code4'].fillna('', inplace=True)
    df['New_Procedure'] = df['Procedure_Code'] + ' ' + df['Modi_Code1'] + ' ' + df['Modi_Code2'] + ' ' +df['Modi_Code3'] + ' ' + df['Modi_Code4']
    df['New_Procedure'] =df['New_Procedure'] .str.strip()
    df.fillna("1", inplace=True)
    return df

In [7]:
def transform_data(df_filtered):
    new_df = df_filtered.copy()
    columns_to_encode = [col for col in df_filtered.columns if col not in ['Amount', 'Amt_Due', 'Days', 'Amount_Approved', 'Date_Entry', 'Process_Date', 'Procedure_Code', 'Claim_Total', 'Claim_No', 'Day_of_Month', 'Weekday']]
    for col in columns_to_encode:
        encoder = joblib.load(f"AllEncodings/{col}_encoding.joblib")
        new_df[col] = df_filtered[col].astype(str).apply(lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1)
    
    return new_df

In [8]:
def predict_amount(df):
    df = df[['Claim_No', 'Billing_Physician', 'Pri_Status', 'Sec_Status', 'Oth_Status', 'Pat_Status', 'Submission_Type',
  'STATE', 'POS', 'InsPayer_Id', 'Insurance_State', 'PLAN_NAME_TYPE', 'DX_Code1', 'DX_Code2', 'DX_Code3', 'PRIMARY_CARE_PHYSICIAN',
    'Provider_Code', 'Practice_Code', 'TAXONOMY_CODE', 'DX_Pointer1', 'InsName_Id', 'DX_Pointer2', 'DX_Pointer3', 'DX_Pointer4', 
    'Modi_Code1', 'Modi_Code2','Modi_Code3', 'Modi_Code4', 'Amount', 'units', 'New_Procedure']]
    
    amount_model = joblib.load("Models/AmountModel.joblib")
    
    predictions = amount_model.predict(df)
    
    return predictions

In [9]:
def predict_days(df):
    df = df[['Billing_Physician','Pri_Status','Sec_Status','Oth_Status','Pat_Status',
    'Submission_Type', 'STATE', 'POS', 'InsPayer_Id','Insurance_State','PLAN_NAME_TYPE','DX_Code1','DX_Code2',
    'DX_Code3','PRIMARY_CARE_PHYSICIAN', 'Provider_Code','Practice_Code', 'TAXONOMY_CODE', 
    'DX_Pointer1','InsName_Id', 'DX_Pointer2', 'DX_Pointer3', 'DX_Pointer4', 'Modi_Code1', 'Modi_Code2', 'Modi_Code3', 'Modi_Code4',
    'units', 'Day_of_Month', 'Weekday', 'New_Procedure']]

    days_model = joblib.load("Models/DaysModel.joblib")

    predictions = days_model.predict(df)
    
    return predictions

In [10]:
def custom_round(value):
        decimal_part = value - np.floor(value)
        if decimal_part >= 0.7:
            return np.ceil(value)
        else:
            return np.floor(value)

In [11]:
def fee_schedule_data():
    
    query = f"""
    quuery
    """
    
    conn, cursor = connection_76()
    df = pd.read_sql(query, conn)
    disconnect(conn, cursor)
    return df

In [12]:
def fee_schedule_mcr_modifier():
    
    query = f"""
    query
    """
    
    conn, cursor = connection_66()
    df = pd.read_sql(query, conn)
    disconnect(conn, cursor)
    return df

In [13]:
def fee_schedule_Current_Year():
    
    query = f"""
    query
    """
    
    conn, cursor = connection_76()
    df = pd.read_sql(query, conn)
    disconnect(conn, cursor)
    return df

In [14]:
start_date = '2025-07-09'
end_date = '2025-07-13'

In [15]:
main_df = fetch_data_from_sql(start_date, end_date)

In [16]:
all_submitted = all_submitted_claims(start_date, end_date)

In [17]:
all_submitted.columns = all_submitted.columns.str.upper()

In [18]:
main_df.shape[0], all_submitted.shape[0]

(515, 174)

In [19]:
main_df.Amount.sum()

np.float64(31489.510000000002)

In [20]:
main_df.Claim_No.nunique(), all_submitted.CLAIM_NO.nunique()

(148, 171)

In [21]:
all_submitted['PRI_STATUS'] = all_submitted['PRI_STATUS'].str.upper()

In [22]:
all_submitted_new_bil = all_submitted[(all_submitted['PRI_STATUS'].isin(['N'])) & ~(all_submitted['SUBMISSION_TYPE'].isin(['QRD', 'DME']))]

In [23]:
all_submitted_new_bil.CLAIM_NO.nunique()

151

In [24]:
main_df.Claim_No.nunique(), main_df.Process_Date.nunique()

(148, 4)

In [25]:
claims = all_submitted_new_bil.CLAIM_NO.unique().tolist()

excluded_claims = [x for x in claims if x not in main_df['Claim_No'].tolist()]

In [26]:
processed_df = preprocess_data(main_df)

In [27]:
transformed_df = transform_data(processed_df)

In [1]:
# transformed_df.head()

In [29]:
Amount_prediction = predict_amount(transformed_df)

In [30]:
Days_prediction = predict_days(transformed_df)

In [31]:
len(Amount_prediction)

515

In [32]:
processed_df['Predicted_Amount'] = Amount_prediction
processed_df['Predicted_Days'] = Days_prediction

In [33]:
processed_df['Predicted_Days'] = processed_df['Predicted_Days'].apply(custom_round).astype(int)

In [34]:
processed_df['Process_Date'] = pd.to_datetime(processed_df['Process_Date'])  
processed_df['Predicted_date'] = processed_df['Process_Date'] + processed_df['Predicted_Days'].apply(lambda x: timedelta(days=x))

In [35]:
main_df['Predictions'] = Amount_prediction
main_df['Predicted_Days'] = Days_prediction
main_df['Predicted_Days'] = main_df['Predicted_Days'].apply(custom_round).astype(int)
main_df['Process_Date'] = pd.to_datetime(main_df['Process_Date'])  
main_df['Predicted_Date'] = main_df['Process_Date'] + main_df['Predicted_Days'].apply(lambda x: timedelta(days=x))
main_df['Created_Date'] = datetime.datetime.now().strftime('%Y-%m-%d')
main_df.drop(['Day_of_Month', 'Weekday', 'Weekday_Name'], axis = 1, inplace = True)

In [36]:
main_df.EMC_NO.dtype

dtype('O')

In [37]:
ds_claim = deepcopy(main_df)

In [38]:
ds_claim['EMC_NO'] = ds_claim['EMC_NO'].fillna('')

In [39]:
ds_claim_new = ds_claim.drop_duplicates(subset={'Claim_No','Billing_Physician', 'EMC_NO', 'Process_Date','InsPayer_Id','Procedure_Code'})

In [40]:
ds_claim_new.shape[0]        

512

In [41]:
ds_claim_new.Process_Date.unique()

<DatetimeArray>
['2025-07-09 00:00:00', '2025-07-10 00:00:00', '2025-07-11 00:00:00',
 '2025-07-13 00:00:00']
Length: 4, dtype: datetime64[ns]

# Adjustment Part 

In [42]:
ds_claim.shape[0]

515

In [43]:
ds_claim.EMC_NO.unique()

array(['12402', '00087', '00001', '60054', '87726', 'UMR', '54763',
       'HNJH', 'WCHP', '22326', 'AMGRP', '00002', 'WLPNT', '84367'],
      dtype=object)

In [44]:
ds_claim[ds_claim['EMC_NO'].str.contains('12402')].shape[0]

95

In [45]:
ppd = pd.read_csv("abc.csv")
ppd_2025 = pd.read_csv("abc_2025.csv")

In [46]:
ppd_modifier = pd.read_csv("data.csv")
ppd_modifier_2025 = pd.read_csv("data_2025.csv")

In [2]:
# ppd.head(2)

In [3]:
# ppd_2025.head(2)

In [4]:
# ppd_modifier.head(2)

In [5]:
# ppd_modifier_2025.head(2)

In [51]:
fee_sched_curr_year = fee_schedule_Current_Year()

In [53]:
fee_schedule = fee_schedule_data()

fee_schedule.drop(columns={'INSPAYER_DESCRIPTION','PRACTICE_CODE'}, inplace=True)
fee_schedule.rename(columns={'BILLING_PHYSICIAN':'Billing_Physician', 'PROCEDURE_CODE':'Procedure_Code', 
                             'INSPAYER_ID':'InsPayer_Id', 'FeeSchedule':'Avg_Approved_Amt'}, inplace=True)

In [54]:
fee_mcr_modifier = fee_schedule_mcr_modifier()

fee_mcr_modifier.drop(columns={'INSPAYER_DESCRIPTION','PRACTICE_CODE'}, inplace=True)
fee_mcr_modifier.rename(columns={'BILLING_PHYSICIAN':'Billing_Physician', 'PROCEDURE_CODE':'Procedure_Code', 
                             'INSPAYER_ID':'InsPayer_Id', 'FeeSchedule':'Avg_Approved_Amt'}, inplace=True)

In [55]:
fee_schedule['EMC_NO'].replace('NONE', 'NA', inplace=True)
fee_schedule['EMC_NO'].replace('', 'NA', inplace=True)

In [56]:
fee_mcr_modifier['EMC_NO'].replace('NONE', 'NA', inplace=True)
fee_mcr_modifier['EMC_NO'].replace('', 'NA', inplace=True)

In [57]:
fee_sched_curr_year['Billing_Physician'] = fee_sched_curr_year['Billing_Physician'].astype(str).str.strip().astype(int)
fee_sched_curr_year['Plan_Name'] = fee_sched_curr_year['Plan_Name'].astype(str).str.strip().str.upper()
fee_sched_curr_year['Procedure_Code'] = fee_sched_curr_year['Procedure_Code'].astype(str).str.strip().str.upper()
fee_sched_curr_year['InsPayer_Id'] = fee_sched_curr_year['InsPayer_Id'].astype(str).str.strip().astype(int)

In [58]:
ppd['Provider_Code'] = ppd['Provider_Code'].astype(str).str.strip().astype(int)
ppd['Plan_Name'] = ppd['Plan_Name'].astype(str).str.strip().str.upper()
ppd['EMC_NO'] = ppd['EMC_NO'].astype(str).str.strip().str.upper()
ppd['Paid_Proc_Code'] = ppd['Paid_Proc_Code'].astype(str).str.strip().str.upper()
ppd['Insurance_State'] = ppd['Insurance_State'].astype(str).str.strip().str.upper()
ppd['InsPayer_State'] = ppd['InsPayer_State'].astype(str).str.strip().str.upper()
ppd['PROVIDER_STATE'] = ppd['PROVIDER_STATE'].astype(str).str.strip().str.upper()
ppd['InsPayer_Id'] = ppd['InsPayer_Id'].astype(str).str.strip().astype(int)

In [59]:
ppd_2025['Provider_Code'] = ppd_2025['Provider_Code'].astype(str).str.strip().astype(int)
ppd_2025['Plan_Name'] = ppd_2025['Plan_Name'].astype(str).str.strip().str.upper()
ppd_2025['EMC_NO'] = ppd_2025['EMC_NO'].astype(str).str.strip().str.upper()
ppd_2025['Paid_Proc_Code'] = ppd_2025['Paid_Proc_Code'].astype(str).str.strip().str.upper()
ppd_2025['Insurance_State'] = ppd_2025['Insurance_State'].astype(str).str.strip().str.upper()
ppd_2025['InsPayer_State'] = ppd_2025['InsPayer_State'].astype(str).str.strip().str.upper()
ppd_2025['PROVIDER_STATE'] = ppd_2025['PROVIDER_STATE'].astype(str).str.strip().str.upper()
ppd_2025['InsPayer_Id'] = ppd_2025['InsPayer_Id'].astype(str).str.strip().astype(int)

In [60]:
ppd_modifier['Provider_Code'] = ppd_modifier['Provider_Code'].astype(str).str.strip().astype(int)
ppd_modifier['Plan_Name'] = ppd_modifier['Plan_Name'].astype(str).str.strip().str.upper()
ppd_modifier['EMC_NO'] = ppd_modifier['EMC_NO'].astype(str).str.strip().str.upper()
ppd_modifier['Paid_Proc_Code'] = ppd_modifier['Paid_Proc_Code'].astype(str).str.strip().str.upper()
ppd_modifier['Insurance_State'] = ppd_modifier['Insurance_State'].astype(str).str.strip().str.upper()
ppd_modifier['InsPayer_State'] = ppd_modifier['InsPayer_State'].astype(str).str.strip().str.upper()
ppd_modifier['PROVIDER_STATE'] = ppd_modifier['PROVIDER_STATE'].astype(str).str.strip().str.upper()
ppd_modifier['InsPayer_Id'] = ppd_modifier['InsPayer_Id'].astype(str).str.strip().astype(int)
ppd_modifier['Modi_Code1'] = ppd_modifier['Modi_Code1'].astype(str).str.strip().str.upper()

In [61]:
ppd_modifier_2025['Provider_Code'] = ppd_modifier_2025['Provider_Code'].astype(str).str.strip().astype(int)
ppd_modifier_2025['Plan_Name'] = ppd_modifier_2025['Plan_Name'].astype(str).str.strip().str.upper()
ppd_modifier_2025['EMC_NO'] = ppd_modifier_2025['EMC_NO'].astype(str).str.strip().str.upper()
ppd_modifier_2025['Paid_Proc_Code'] = ppd_modifier_2025['Paid_Proc_Code'].astype(str).str.strip().str.upper()
ppd_modifier_2025['Insurance_State'] = ppd_modifier_2025['Insurance_State'].astype(str).str.strip().str.upper()
ppd_modifier_2025['InsPayer_State'] = ppd_modifier_2025['InsPayer_State'].astype(str).str.strip().str.upper()
ppd_modifier_2025['PROVIDER_STATE'] = ppd_modifier_2025['PROVIDER_STATE'].astype(str).str.strip().str.upper()
ppd_modifier_2025['InsPayer_Id'] = ppd_modifier_2025['InsPayer_Id'].astype(str).str.strip().astype(int)
ppd_modifier_2025['Modi_Code1'] = ppd_modifier_2025['Modi_Code1'].astype(str).str.strip().str.upper()

In [62]:
ds_claim['Provider_Code'] = ds_claim['Provider_Code'].astype(str).str.strip().astype(int)
ds_claim['Plan_Name'] = ds_claim['Plan_Name'].astype(str).str.strip().str.upper()
ds_claim['EMC_NO'] = ds_claim['EMC_NO'].astype(str).str.strip().str.upper()
ds_claim['Procedure_Code'] = ds_claim['Procedure_Code'].astype(str).str.strip().str.upper()
ds_claim['Insurance_State'] = ds_claim['Insurance_State'].astype(str).str.strip().str.upper()
ds_claim['InsPayer_State'] = ds_claim['InsPayer_State'].astype(str).str.strip().str.upper()
ds_claim['PROVIDER_STATE'] = ds_claim['PROVIDER_STATE'].astype(str).str.strip().str.upper()
ds_claim['InsPayer_Id'] = ds_claim['InsPayer_Id'].astype(str).str.strip().astype(int)
ds_claim['Modi_Code1'] = ds_claim['Modi_Code1'].astype(str).str.strip().str.upper()

In [63]:
fee_schedule['Billing_Physician'] = fee_schedule['Billing_Physician'].astype(str).str.strip().astype(int)
fee_schedule['InsPayer_Id'] = fee_schedule['InsPayer_Id'].astype(str).str.strip().astype(int)
fee_schedule['Procedure_Code'] = fee_schedule['Procedure_Code'].astype(str).str.strip().str.upper()
fee_schedule['EMC_NO'] = fee_schedule['EMC_NO'].astype(str).str.strip().str.upper()

In [64]:
fee_mcr_modifier['Billing_Physician'] = fee_mcr_modifier['Billing_Physician'].astype(str).str.strip().astype(int)
fee_mcr_modifier['InsPayer_Id'] = fee_mcr_modifier['InsPayer_Id'].astype(str).str.strip().astype(int)
fee_mcr_modifier['Procedure_Code'] = fee_mcr_modifier['Procedure_Code'].astype(str).str.strip().str.upper()
fee_mcr_modifier['EMC_NO'] = fee_mcr_modifier['EMC_NO'].astype(str).str.strip().str.upper()

In [65]:
ppd.rename(columns={"Paid_Proc_Code":"Procedure_Code","Provider_Code":"Billing_Physician"}, inplace=True)
ppd_2025.rename(columns={"Paid_Proc_Code":"Procedure_Code","Provider_Code":"Billing_Physician"}, inplace=True)

In [66]:
ppd_modifier.rename(columns={"Paid_Proc_Code":"Procedure_Code","Provider_Code":"Billing_Physician"}, inplace=True)
ppd_modifier_2025.rename(columns={"Paid_Proc_Code":"Procedure_Code","Provider_Code":"Billing_Physician"}, inplace=True)

In [67]:
day_pro_EMC_stat_provstat = ppd.groupby(['Billing_Physician', 'EMC_NO', 'Insurance_State', 'PROVIDER_STATE']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

day_pro_EMC_stat = ppd.groupby(['Billing_Physician', 'EMC_NO', 'Insurance_State']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

day_pro_EMC = ppd.groupby(['Billing_Physician', 'EMC_NO']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

day_EMC = ppd.groupby(['EMC_NO']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()


In [68]:
day_pro_ins_stat_provstat = ppd.groupby(['Billing_Physician', 'InsPayer_Id', 'Insurance_State', 'PROVIDER_STATE']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

day_pro_ins_stat = ppd.groupby(['Billing_Physician', 'InsPayer_Id', 'Insurance_State']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

day_pro_ins = ppd.groupby(['Billing_Physician', 'InsPayer_Id']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

day_ins = ppd.groupby(['InsPayer_Id']).agg(
    Days=('Days', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()


In [69]:
#2025 Modifier Data

amt_pro_plan_emc_modifier_cpt_2025 = ppd_modifier_2025.groupby(['Billing_Physician','Plan_Name','EMC_NO','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_plan_InsPayer_modifier_cpt_2025 = ppd_modifier_2025.groupby(['Billing_Physician','Plan_Name','InsPayer_Id','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_emc_modifier_proc_2025 = ppd_modifier_2025.groupby(['Billing_Physician','EMC_NO','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_InsPayer_modifier_proc_2025 = ppd_modifier_2025.groupby(['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()



In [70]:
amt_pro_plan_emc_modifier_cpt_2025 = amt_pro_plan_emc_modifier_cpt_2025[amt_pro_plan_emc_modifier_cpt_2025['Count']>=3]
amt_pro_plan_InsPayer_modifier_cpt_2025 = amt_pro_plan_InsPayer_modifier_cpt_2025[amt_pro_plan_InsPayer_modifier_cpt_2025['Count']>=3]
amt_pro_emc_modifier_proc_2025 = amt_pro_emc_modifier_proc_2025[amt_pro_emc_modifier_proc_2025['Count']>=3]
amt_pro_InsPayer_modifier_proc_2025 = amt_pro_InsPayer_modifier_proc_2025[amt_pro_InsPayer_modifier_proc_2025['Count']>=3]

In [71]:
amt_pro_plan_emc_modifier_cpt = ppd_modifier.groupby(['Billing_Physician','Plan_Name','EMC_NO','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_plan_InsPayer_modifier_cpt = ppd_modifier.groupby(['Billing_Physician','Plan_Name','InsPayer_Id','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_emc_modifier_proc = ppd_modifier.groupby(['Billing_Physician','EMC_NO','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_InsPayer_modifier_proc = ppd_modifier.groupby(['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

In [72]:
#2025 Data

amt_pro_plan_emc_cpt_2025 = ppd_2025.groupby(['Billing_Physician', 'Plan_Name', 'EMC_NO', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_plan_InsPayer_cpt_2025 = ppd_2025.groupby(['Billing_Physician', 'Plan_Name', 'InsPayer_Id', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_emc_proc_2025 = ppd_2025.groupby(['Billing_Physician', 'EMC_NO', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_InsPayer_proc_2025 = ppd_2025.groupby(['Billing_Physician', 'InsPayer_Id', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_proc_2025 = ppd_2025.groupby(['Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_cpt_2025 = ppd_2025.groupby(['Billing_Physician', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', 'median'),
    Count=('Claim_No', 'nunique')
    ).reset_index()

In [73]:
samt_pro_plan_emc_cpt_2025 = amt_pro_plan_emc_cpt_2025[amt_pro_plan_emc_cpt_2025['Count']>=3]
amt_pro_plan_InsPayer_cpt_2025 = amt_pro_plan_InsPayer_cpt_2025[amt_pro_plan_InsPayer_cpt_2025['Count']>=3]
amt_pro_emc_proc_2025 = amt_pro_emc_proc_2025[amt_pro_emc_proc_2025['Count']>=3]
amt_pro_InsPayer_proc_2025 = amt_pro_InsPayer_proc_2025[amt_pro_InsPayer_proc_2025['Count']>=3]
amt_proc_2025 = amt_proc_2025[amt_proc_2025['Count']>=3]
amt_pro_cpt_2025 = amt_pro_cpt_2025[amt_pro_cpt_2025['Count']>=3]

In [74]:
amt_pro_plan_emc_cpt = ppd.groupby(['Billing_Physician', 'Plan_Name', 'EMC_NO', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_plan_InsPayer_cpt = ppd.groupby(['Billing_Physician', 'Plan_Name', 'InsPayer_Id', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_emc_proc = ppd.groupby(['Billing_Physician', 'EMC_NO', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_InsPayer_proc = ppd.groupby(['Billing_Physician', 'InsPayer_Id', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_pro_cpt = ppd.groupby(['Billing_Physician', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', 'median'),
    Count=('Claim_No', 'nunique')
    ).reset_index()

In [75]:
amt_spec_emc_state_cpt = ppd.groupby(['SPECIALIZATION_CODE', 'EMC_NO', 'PROVIDER_STATE', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_spec_InsPayer_state_cpt = ppd.groupby(['SPECIALIZATION_CODE', 'InsPayer_Id', 'PROVIDER_STATE', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_spec_state_cpt = ppd.groupby(['SPECIALIZATION_CODE', 'PROVIDER_STATE', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_state_cpt = ppd.groupby(['PROVIDER_STATE', 'Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()

amt_proc = ppd.groupby(['Procedure_Code']).agg(
    Avg_Approved_Amt=('Avg_Approved_Amt', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    Count=('Claim_No', 'nunique')
    ).reset_index()


In [76]:
ds_claim['Predicted_Date'] = pd.to_datetime(ds_claim['Predicted_Date'])
ds_claim['Process_Date'] = pd.to_datetime(ds_claim['Process_Date'])

In [77]:
# Calculate the minimum Predicted_Date for each group"
ds_claim['Predicted_Date'] = ds_claim.groupby(['Claim_No', 'Process_Date', 'InsPayer_Id'])['Predicted_Date'].transform('min')

In [78]:
ds_claim['Predicted_Days'] = ds_claim.groupby(['Claim_No', 'Process_Date', 'InsPayer_Id'])['Predicted_Days'].transform('min')

In [79]:
amt_pro_plan_emc_cpt['Billing_Physician'] = amt_pro_plan_emc_cpt['Billing_Physician'].astype(int)
amt_pro_plan_InsPayer_cpt['Billing_Physician'] = amt_pro_plan_InsPayer_cpt['Billing_Physician'].astype(int)
amt_pro_emc_proc['Billing_Physician'] = amt_pro_emc_proc['Billing_Physician'].astype(int)
amt_pro_InsPayer_proc['Billing_Physician'] = amt_pro_InsPayer_proc['Billing_Physician'].astype(int)

In [80]:
fee_schedule_emc = fee_schedule[['Billing_Physician','EMC_NO','Procedure_Code','Avg_Approved_Amt']]
fee_schedule_ins_payer = fee_schedule[['Billing_Physician','InsPayer_Id','Procedure_Code','Avg_Approved_Amt']]
fee_schedule_emc = fee_schedule_emc[fee_schedule_emc['EMC_NO'] != 'NA']

In [81]:
fee_mcr_modifier_emc = fee_mcr_modifier[['Billing_Physician','EMC_NO','Procedure_Code','Modi_Code1','Avg_Approved_Amt']]
fee_mcr_modifier_ins_payer = fee_mcr_modifier[['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1','Avg_Approved_Amt']]
fee_mcr_modifier_emc = fee_mcr_modifier[fee_mcr_modifier['EMC_NO'] != 'NA']

In [82]:
fee_mcr_modifier_emc.drop(columns={'InsPayer_Id'}, inplace=True)

In [83]:
ds_claim[(ds_claim['InsPayer_Id'] == 5002492) & (ds_claim['Modi_Code1'].notna()) 
         & (ds_claim['Modi_Code1'] !='')]['Modi_Code1'].unique()

array(['59', '25', 'LT', 'NONE', '50'], dtype=object)

In [84]:
fee_mcr_modifier_emc.Modi_Code1.unique()

array(['53', 'TC', '26'], dtype=object)

# Layered Approach (Amount)

In [85]:
# --- Part 1: Process _merge == 'both' with custom logic
def resolve_common_row(row):
    # Include key columns explicitly
    result = row[key_cols].copy()
    
    if row['Count_1'] < 5 and row['Count_2'] >= 2 * row['Count_1']:
        # Add values from dataframe 2
        selected_cols = [col for col in row.index if col.endswith('_2')]
        temp = row[selected_cols].rename(lambda x: x[:-2])
    else:
        # Add values from dataframe 1
        selected_cols = [col for col in row.index if col.endswith('_1')]
        temp = row[selected_cols].rename(lambda x: x[:-2])
    
    # Merge selected values with key columns
    return pd.concat([result, temp])

In [86]:
def common_left_right_df(df):
    # Apply to rows where both sides exist
    merged_common = df[df['_merge'] == 'both'].apply(resolve_common_row, axis=1)

    # --- Part 2: Rows only in both_merge1
    merged_left_only = df[df['_merge'] == 'left_only']
    merged_left_only = merged_left_only[[col for col in merged_left_only.columns if col.endswith('_1') or col in key_cols]]
    merged_left_only.columns = [col[:-2] if col.endswith('_1') else col for col in merged_left_only.columns]

    # --- Part 3: Rows only in both_merge2
    merged_right_only = df[df['_merge'] == 'right_only']
    merged_right_only = merged_right_only[[col for col in merged_right_only.columns if col.endswith('_2') or col in key_cols]]
    merged_right_only.columns = [col[:-2] if col.endswith('_2') else col for col in merged_right_only.columns]

    # --- Combine df_common and df_left_only if you want them together:
    merged_common_left = pd.concat([merged_common, merged_left_only], ignore_index=True)

    # Optional: Ensure 'Count' is integer
    merged_common_left['Count'] = merged_common_left['Count'].astype(int)
    merged_right_only['Count'] = merged_right_only['Count'].astype(int)
    
    return merged_common_left, merged_right_only

In [87]:
def merge_both_dfs(df1, df2):
    # Outer merge to track origin
    merged = pd.merge(
        df1, df2,
        on=key_cols,
        how='outer',
        suffixes=('_1', '_2'),
        indicator=True
    )
    
    return merged

In [88]:
def ds_claim_remaining(main_df, new_df):
    # Use merge with indicator to find rows in ds_claim NOT in df_common_and_left_only
    unmatched = pd.merge(
        main_df,
        new_df[key_cols],
        on=key_cols,
        how='left',
        indicator=True
    )

    # Filter rows that were only in ds_claim
    ds_remaining = unmatched[unmatched['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    return ds_remaining

In [89]:
# ds_claim.head(2)

In [90]:
# fee_mcr_modifier_emc.head(2)

In [91]:
key_cols = ['Claim_No', 'Billing_Physician', 'Process_Date', 'InsPayer_Id', 'Procedure_Code']
ds_claim = ds_claim.drop_duplicates(subset=key_cols)

In [92]:
ds_claim.shape[0]

512

In [93]:
fee_mcr_modifier_ins_payer.dtypes

Billing_Physician      int64
InsPayer_Id            int64
Procedure_Code        object
Modi_Code1            object
Avg_Approved_Amt     float64
dtype: object

In [94]:
ds_claim[['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1']].dtypes

Billing_Physician     int64
InsPayer_Id           int64
Procedure_Code       object
Modi_Code1           object
dtype: object

# Step 1

In [96]:
# Step 1: Merge ds_claim with amt_pro_plan_InsPayer_cpt using LEFT join and indicator
medicare_modi_merge = ds_claim.merge(
    fee_mcr_modifier_ins_payer,
    on=['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
medicare_modi_merge_both = medicare_modi_merge[medicare_modi_merge['_merge'] == 'both']
rem_ds_claim_n = medicare_modi_merge[medicare_modi_merge['_merge'] == 'left_only']

medicare_modi_merge.drop(columns=['_merge'], inplace=True)
medicare_modi_merge_both.drop(columns=['_merge'], inplace=True)
rem_ds_claim_n.drop(columns=['_merge','Avg_Approved_Amt'], inplace=True)

In [97]:
medicare_modi_merge_both.shape[0]

0

In [98]:
medicare_modi_merge_both['LAYER_INFO'] = 'fee_mcr_modifier_ins_payer'

In [99]:
rem_ds_claim_n.shape[0]

512

# Step 2

In [100]:
# Step 1: Merge ds_claim with amt_pro_plan_InsPayer_cpt using LEFT join and indicator
medicare_modi_merge_inspayer = rem_ds_claim_n.merge(
    fee_mcr_modifier_emc,
    on=['Billing_Physician','EMC_NO','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
medicare_modi_merge_inspayer_both = medicare_modi_merge_inspayer[medicare_modi_merge_inspayer['_merge'] == 'both']
rem_ds_claim_nw = medicare_modi_merge_inspayer[medicare_modi_merge_inspayer['_merge'] == 'left_only']

medicare_modi_merge_inspayer.drop(columns=['_merge'], inplace=True)
medicare_modi_merge_inspayer_both.drop(columns=['_merge'], inplace=True)
rem_ds_claim_nw.drop(columns=['_merge','Avg_Approved_Amt'], inplace=True)

In [101]:
medicare_modi_merge_inspayer_both.shape[0]

0

In [102]:
medicare_modi_merge_inspayer_both['LAYER_INFO'] = 'fee_mcr_modifier_emc'

# Step 3 (Fee Schedule Medicare)

In [103]:
ds_claim.shape[0], rem_ds_claim_nw.shape[0]

(512, 512)

In [104]:
modi_merge4 = rem_ds_claim_nw.merge(
    fee_schedule_ins_payer, 
    on=['Billing_Physician', 'InsPayer_Id', 'Procedure_Code'], 
    how='left', 
    indicator=True
)

merged_common_left3 = modi_merge4[
    (modi_merge4['_merge'] == 'both') & 
    (modi_merge4['InsPayer_Id'] == 5002492)
]

In [105]:
ds_remaining3 = modi_merge4[
    (modi_merge4['_merge'] == 'left_only') |
    (
        (modi_merge4['_merge'] == 'both') &
        (modi_merge4['InsPayer_Id'] != 5002492)
    )
]

ds_remaining3.drop(columns={'Avg_Approved_Amt','_merge'}, inplace=True)

In [106]:
merged_common_left3.shape[0],ds_remaining3.shape[0]

(39, 473)

In [107]:
merged_common_left3['LAYER_INFO'] = 'fee_schedule_ins_payer'

# Step 4 (Fee Schedule Medicare)

In [108]:
modi_merge5 = ds_remaining3.merge(
    fee_schedule_emc, 
    on=['Billing_Physician', 'EMC_NO', 'Procedure_Code'], 
    how='left', 
    indicator=True
)

In [109]:
modi_merge5 = modi_merge5.drop_duplicates(subset={'Claim_No','Billing_Physician', 'EMC_NO', 'Process_Date','InsPayer_Id','Procedure_Code'})
merged_common_left4 = modi_merge5[(modi_merge5['_merge']=='both') & (modi_merge5['EMC_NO']=='12402')]

In [110]:
ds_remaining4 = modi_merge5[(modi_merge5['_merge'] == 'left_only') | ((modi_merge5['_merge'] == 'both')&(modi_merge5['EMC_NO'] != '12402'))]
ds_remaining4.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [111]:
merged_common_left4['LAYER_INFO'] = 'fee_schedule_emc'

In [112]:
modi_merge5.shape[0], merged_common_left4.shape[0], ds_remaining4.shape[0]

(473, 0, 473)

# Modifier 2025

# Step 5

In [113]:
ds_remaining4.shape[0]

473

In [114]:
modi_merge_25 = ds_remaining4.merge(
    amt_pro_plan_InsPayer_modifier_cpt_2025,
    on=['Billing_Physician','Plan_Name','InsPayer_Id','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge_both_2025 = modi_merge_25[modi_merge_25['_merge'] == 'both']

ds_remaining2025 = modi_merge_25[modi_merge_25['_merge'] == 'left_only']
ds_remaining2025.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge_25.drop(columns=['_merge'], inplace=True)
modi_merge_both_2025.drop(columns=['_merge'], inplace=True)

In [115]:
modi_merge_both_2025['LAYER_INFO'] = 'amt_pro_plan_InsPayer_modifier_cpt_2025'

In [116]:
ds_remaining2025.drop(columns={'Count'}, inplace=True)

In [117]:
ds_remaining2025.shape[0]

473

In [118]:
ds_remaining2025.head(2)

,Claim_No,BILL_DATE,DOS,Patient_Account,Patient_First_Name,Relationship,Patient_Last_Name,Insurance_Id,Att_Physician,Amt_Due,Claim_Total,Pri_Status,Sec_Status,Oth_Status,Pat_Status,Process_Date,Submission_Type,First_lvl_Rejection_Reason,Second_lvl_Rejection_Reason,SUBMISSION_STATUS,CITY,STATE,ZIP,POS,PRI_SEC_OTH_TYPE,SUBMITTED_DX_CODES,SUBMITTED_PROC_CODES,Sent_837_Data,INSURANCE_FILE_ID,Billing_Physician,BILLING_PROV_NPI,PROVIDER_NAME,PATIENT_GENDER,InsName_Id,EMC_NO,InsPayer_Description,InsPayer_State,InsPayer_Id,Insurance_City,Insurance_State,Insurance_Zip,Insurance_CardCategory,Insurance_Department,InActive,ClaimFilingLimit,AppealFilingLimit,STOP_SUBMISSION,IS_DEMO,Claim_Insurance_Id,Policy_Number,Termination_Date,Eligibility_Status,CCN,ICN,Late_Filing,certification_number,rej_code,Plan_Name,PLAN_NAME_TYPE,Co_Insurance,PLAN_ID,Location_Code,Attending_Physician,PRI_STATUS,SEC_STATUS,OTH_STATUS,Supervising_Physician,Referring_Physician,PA_NUMBER,Referral_Number,ICN_Number,Facility_Code,Hospital_From,Hospital_To,Claim_Status_Date,Pri_Ins_Payment,Sec_Ins_Payment,Oth_Ins_Payment,Patient_Payment,Amt_Paid,DX_Code1,DX_Code2,DX_Code3,DX_Code4,DX_Code5,DX_Code6,DX_Code7,DX_Code8,Is_Self_Pay,reference_number,Ordering_Physician,Claims_Status_Code,DX_Code9,DX_Code10,DX_Code11,DX_Code12,DWC_ID,DWC_DETAIL_ID,Advance_Pat_Payment,PRIMARY_CARE_PHYSICIAN,Provider_Code,Provid_FName,Provid_MName,Provid_LName,PROVIDER_STATE,Provid_UPIN,Provid_State_License,License_No,Site_Id,DEA_No,DEA_Expiry_Date,SSN,Provider_Id,Provider_Title,Gender,Is_Billing_Physician,NPI,Old_Billing,Old_Billing_Date,Temination,SPECIALIZATION_CODE,SPI,PROVIDER_IS_ATTENDING_PHYSICIAN,PROVIDER_DISPLAY_NAME,SERVERNAME,LICENSE_STATE,Pointer_setting,IS_BILL_PROVIDER,INV_PROVID_INURANCE,isCancel,isChange,Is_Test_Provider,EXTERNAL_PROVIDER_ID,RESPONSIBLE_TEAM_ID,Practice_Code,Prac_Name,Prac_Type,Prac_State,PRAC_LICENSE_NUMBER,Prac_City,Prac_Zip,Prac_Tax_Id,Patient_Billing,Contract_Version,Agreement_Date,Billing_Percentage,Old_Billing_Percentage,Old_Billing_Percentage_From,Old_Billing_Percentage_To,Client_Billing_Percentage,Is_Active,Mailing_City,Mailing_State,Mailing_Zip,EMR_Name,Is_Practice_Facility_Based,Invoice_Practice_Name,Invoice_Prac_State,Invoice_Prac_City,Invoice_Prac_Zip,TAXONOMY_CODE,Termination_Notice,Termination_Notice_Date,units,claim_charges_id,Sequence_No,Procedure_Code,DOE,DX_Pointer1,DX_Pointer2,DX_Pointer3,DX_Pointer4,Modi_Code1,Modi_Code2,Modi_Code3,Modi_Code4,Amount,Predictions,Predicted_Days,Predicted_Date,Created_Date
0,9283770830,2025-06-17,2025-06-17,10056510139,Hernan,S,VALLEJO JIJON,5341693,100,240.05,225.05,N,,,W,2025-07-09,DS,None,None,PENDING,NORTH BRUNSWICK,NJ,089021404,OF,P,"Z68.26,R94.4,R94.31,I10,N40.1,E78.79,E11.9,E55...","3044F,G8420,G8427,G9903,G8950,G0439,93000",None,None,100,1881648541,MEHNAZ HAQ,MALE,5001996,12402,MEDICARE-NJ,NJ,5002492,MECHANICSBURG,PA,170551802,,Common Address,False,None,None,None,None,9284908034,2RX0F27CV43,None,None,None,,False,None,None,NONE,None,0,None,158,100,B,,,0.0,53511211.0,None,None,None,0.0,None,None,None,0.0,0.0,0.0,0.0,0.0,Z68.26,R94.4,R94.31,I10,N40.1,E78.79,E11.9,E55.9,False,None,0.0,9.0,I25.10,G62.9,Z00.00,,None,None,None,None,100,MEHNAZ,,HAQ,NJ,None,None,25MA06414600,MH01,BH5001134,2017-07-03,060766386,1,MD,FEMALE,True,1881648541,None,1900-01-01,None,011,6475731553003,None,None,TALKEHRDB-SRV,NJ,Y,None,None,True,True,None,None,NaN,100,"MEHNAZ A. HAQ, MD, LLC",I,NJ,MA64146,NORTH BRUNSWICK,089021021,223549493,True,0 . -,2002-04-01,5.0,None,1986-12-07,1989-01-03,None,True,,,,TALKEHR,False,"MEHNAZ A. HAQ, MD, LLC",NJ,NORTH BRUNSWICK,08902,207R00000X,False,1900-01-01,1,9287418378,2,3008F,2025-06-17 11:27:00,1,2,3,4,,,,,5.00,2.194900,93,2025-10-10,2025-07-14
1,9283770830,2025-06-17,2025-06-17,10056510139,Hernan,S,VALLEJO JIJON,5341693,100,240.05,225.05,N,,,W,2025-07-09,DS,None,None,PENDING,NORTH BRUNSWICK,NJ,089021404,OF,P,"Z68.26,R94.4,R94.31,I10,N40.1,E78.79,E11.9,E55...","3044F,G8420,G8427,G9903,G8950,G0439,93

In [119]:
modi_merge_both_2025.shape[0]

0

In [120]:
modi_merge_both_2025.head(2)

,Claim_No,BILL_DATE,DOS,Patient_Account,Patient_First_Name,Relationship,Patient_Last_Name,Insurance_Id,Att_Physician,Amt_Due,Claim_Total,Pri_Status,Sec_Status,Oth_Status,Pat_Status,Process_Date,Submission_Type,First_lvl_Rejection_Reason,Second_lvl_Rejection_Reason,SUBMISSION_STATUS,CITY,STATE,ZIP,POS,PRI_SEC_OTH_TYPE,SUBMITTED_DX_CODES,SUBMITTED_PROC_CODES,Sent_837_Data,INSURANCE_FILE_ID,Billing_Physician,BILLING_PROV_NPI,PROVIDER_NAME,PATIENT_GENDER,InsName_Id,EMC_NO,InsPayer_Description,InsPayer_State,InsPayer_Id,Insurance_City,Insurance_State,Insurance_Zip,Insurance_CardCategory,Insurance_Department,InActive,ClaimFilingLimit,AppealFilingLimit,STOP_SUBMISSION,IS_DEMO,Claim_Insurance_Id,Policy_Number,Termination_Date,Eligibility_Status,CCN,ICN,Late_Filing,certification_number,rej_code,Plan_Name,PLAN_NAME_TYPE,Co_Insurance,PLAN_ID,Location_Code,Attending_Physician,PRI_STATUS,SEC_STATUS,OTH_STATUS,Supervising_Physician,Referring_Physician,PA_NUMBER,Referral_Number,ICN_Number,Facility_Code,Hospital_From,Hospital_To,Claim_Status_Date,Pri_Ins_Payment,Sec_Ins_Payment,Oth_Ins_Payment,Patient_Payment,Amt_Paid,DX_Code1,DX_Code2,DX_Code3,DX_Code4,DX_Code5,DX_Code6,DX_Code7,DX_Code8,Is_Self_Pay,reference_number,Ordering_Physician,Claims_Status_Code,DX_Code9,DX_Code10,DX_Code11,DX_Code12,DWC_ID,DWC_DETAIL_ID,Advance_Pat_Payment,PRIMARY_CARE_PHYSICIAN,Provider_Code,Provid_FName,Provid_MName,Provid_LName,PROVIDER_STATE,Provid_UPIN,Provid_State_License,License_No,Site_Id,DEA_No,DEA_Expiry_Date,SSN,Provider_Id,Provider_Title,Gender,Is_Billing_Physician,NPI,Old_Billing,Old_Billing_Date,Temination,SPECIALIZATION_CODE,SPI,PROVIDER_IS_ATTENDING_PHYSICIAN,PROVIDER_DISPLAY_NAME,SERVERNAME,LICENSE_STATE,Pointer_setting,IS_BILL_PROVIDER,INV_PROVID_INURANCE,isCancel,isChange,Is_Test_Provider,EXTERNAL_PROVIDER_ID,RESPONSIBLE_TEAM_ID,Practice_Code,Prac_Name,Prac_Type,Prac_State,PRAC_LICENSE_NUMBER,Prac_City,Prac_Zip,Prac_Tax_Id,Patient_Billing,Contract_Version,Agreement_Date,Billing_Percentage,Old_Billing_Percentage,Old_Billing_Percentage_From,Old_Billing_Percentage_To,Client_Billing_Percentage,Is_Active,Mailing_City,Mailing_State,Mailing_Zip,EMR_Name,Is_Practice_Facility_Based,Invoice_Practice_Name,Invoice_Prac_State,Invoice_Prac_City,Invoice_Prac_Zip,TAXONOMY_CODE,Termination_Notice,Termination_Notice_Date,units,claim_charges_id,Sequence_No,Procedure_Code,DOE,DX_Pointer1,DX_Pointer2,DX_Pointer3,DX_Pointer4,Modi_Code1,Modi_Code2,Modi_Code3,Modi_Code4,Amount,Predictions,Predicted_Days,Predicted_Date,Created_Date,Avg_Approved_Amt,Count,LAYER_INFO


## Step 6

In [121]:
modi_merge_25_n = ds_remaining2025.merge(
    amt_pro_plan_emc_modifier_cpt_2025,
    on=['Billing_Physician','Plan_Name','EMC_NO','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge_both_2025_n = modi_merge_25_n[modi_merge_25_n['_merge'] == 'both']

ds_remaining2025_n = modi_merge_25_n[modi_merge_25_n['_merge'] == 'left_only']
ds_remaining2025_n.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge_25_n.drop(columns=['_merge'], inplace=True)
modi_merge_both_2025_n.drop(columns=['_merge'], inplace=True)

In [122]:
modi_merge_both_2025_n['LAYER_INFO'] = 'amt_pro_plan_emc_modifier_cpt_2025'

In [123]:
ds_remaining2025_n.drop(columns={'Count'}, inplace=True)

In [124]:
modi_merge_both_2025_n.shape[0]

0

In [125]:
ds_remaining2025_n.shape[0]

473

In [126]:
amt_pro_emc_modifier_proc_2025.head(2)

,Billing_Physician,EMC_NO,Procedure_Code,Modi_Code1,Avg_Approved_Amt,Count
1,100,00001,99213,25,28.25,7
2,100,00001,99213,95,56.50,15


## Step 7

In [127]:
modi_merge_25_n1 = ds_remaining2025_n.merge(
    amt_pro_InsPayer_modifier_proc_2025,
    on=['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge_both_2025_n1 = modi_merge_25_n1[modi_merge_25_n1['_merge'] == 'both']

ds_remaining2025_n1 = modi_merge_25_n1[modi_merge_25_n1['_merge'] == 'left_only']
ds_remaining2025_n1.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge_25_n1.drop(columns=['_merge'], inplace=True)
modi_merge_both_2025_n1.drop(columns=['_merge'], inplace=True)

ds_remaining2025_n1.drop(columns={'Count'}, inplace=True)

In [128]:
modi_merge_both_2025_n1['LAYER_INFO'] = 'amt_pro_InsPayer_modifier_proc_2025'

In [129]:
modi_merge_both_2025_n1.shape[0]

13

In [130]:
ds_remaining2025_n1.shape[0]

460

## Step 8

In [131]:
modi_merge_25_n2 = ds_remaining2025_n1.merge(
    amt_pro_emc_modifier_proc_2025,
    on=['Billing_Physician','EMC_NO','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge_both_2025_n2 = modi_merge_25_n2[modi_merge_25_n2['_merge'] == 'both']

ds_remaining2025_n2 = modi_merge_25_n2[modi_merge_25_n2['_merge'] == 'left_only']
ds_remaining2025_n2.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge_25_n2.drop(columns=['_merge'], inplace=True)
modi_merge_both_2025_n2.drop(columns=['_merge'], inplace=True)

ds_remaining2025_n2.drop(columns={'Count'}, inplace=True)

In [132]:
modi_merge_both_2025_n2['LAYER_INFO'] = 'amt_pro_emc_modifier_proc_2025'

In [133]:
ds_remaining2025_n2.shape[0]

455

In [134]:
modi_merge_both_2025_n2.shape[0]

5

# Step 9

In [135]:
ds_remaining2025_n2.shape[0]

455

In [136]:
# rem_ds_claim_nw.head()

In [137]:
# Step 1: Merge ds_claim with amt_pro_plan_InsPayer_cpt using LEFT join and indicator
modi_merge = ds_remaining2025_n2.merge(
    amt_pro_plan_InsPayer_modifier_cpt,
    on=['Billing_Physician','Plan_Name','InsPayer_Id','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge_both = modi_merge[modi_merge['_merge'] == 'both']

modi_merge.drop(columns=['_merge'], inplace=True)
modi_merge_both.drop(columns=['_merge'], inplace=True)

In [138]:
modi_merge_both['LAYER_INFO'] = 'amt_pro_plan_InsPayer_modifier_cpt'

# Step 10

In [139]:
modi_merge1 = ds_remaining2025_n2.merge(
    amt_pro_plan_emc_modifier_cpt,
    on=['Billing_Physician','Plan_Name','EMC_NO','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)

modi_merge_both1 = modi_merge1[modi_merge1['_merge'] == 'both']

modi_merge1.drop(columns=['_merge'], inplace=True)
modi_merge_both1.drop(columns=['_merge'], inplace=True)

In [140]:
modi_merge_both1['LAYER_INFO'] = 'amt_pro_plan_emc_modifier_cpt'

In [141]:
modi_merge_both.shape

(0, 184)

In [142]:
modi_merge_both1.shape

(0, 184)

In [143]:
modi_merge_both.Claim_No.nunique(), modi_merge_both1.Claim_No.nunique()

(0, 0)

In [144]:
merged_both = merge_both_dfs(modi_merge_both, modi_merge_both1)

In [145]:
merged_both.shape[0]

0

In [146]:
merged_common_left, merged_right_only = common_left_right_df(merged_both)

In [147]:
merged_common_left.shape, merged_right_only.shape

((0, 543), (0, 184))

In [148]:
ds_remaining = ds_claim_remaining(ds_remaining2025_n2, merged_common_left)

In [149]:
ds_remaining.shape

(455, 181)

## Step 11

In [150]:
modi_merge2 = ds_remaining.merge(
    amt_pro_InsPayer_modifier_proc,
    on=['Billing_Physician','InsPayer_Id','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)

modi_merge_both2 = modi_merge2[modi_merge2['_merge'] == 'both']

modi_merge2.drop(columns=['_merge'], inplace=True)
modi_merge_both2.drop(columns=['_merge'], inplace=True)

In [151]:
modi_merge_both2['LAYER_INFO'] = 'amt_pro_InsPayer_modifier_proc'

In [152]:
modi_merge_both2.shape

(2, 184)

In [153]:
merged_both1 = merge_both_dfs(merged_right_only, modi_merge_both2)

In [154]:
merged_both1.shape[0]

2

In [155]:
merged_common_left1, merged_right_only1 = common_left_right_df(merged_both1)

In [156]:
merged_common_left1.shape[0], merged_right_only1.shape[0]

(0, 2)

In [157]:
ds_remaining1 = ds_claim_remaining(ds_remaining, merged_common_left1)

In [158]:
ds_remaining1.shape

(455, 181)

## Step 12

In [159]:
modi_merge3 = ds_remaining1.merge(
    amt_pro_emc_modifier_proc,
    on=['Billing_Physician','EMC_NO','Procedure_Code','Modi_Code1'],
    how='left',
    indicator=True
)

modi_merge_both3 = modi_merge3[modi_merge3['_merge'] == 'both']

modi_merge3.drop(columns=['_merge'], inplace=True)
modi_merge_both3.drop(columns=['_merge'], inplace=True)

In [160]:
modi_merge_both3['LAYER_INFO'] = 'amt_pro_emc_modifier_proc'

In [161]:
merged_both2 = merge_both_dfs(merged_right_only1, modi_merge_both3)

In [163]:
merged_both2.shape[0],modi_merge_both3.shape[0]

(2, 2)

In [164]:
merged_common_left2, merged_right_only2 = common_left_right_df(merged_both2)

In [165]:
merged_common_left2.shape[0], merged_right_only2.shape[0]

(2, 0)

In [166]:
ds_remaining2 = ds_claim_remaining(ds_remaining1, merged_common_left2)

In [167]:
ds_remaining2.shape[0]

453

# Provider plan 2025

## Step 13

In [169]:
modi_merge3_2025 = ds_remaining2.merge(
    amt_pro_plan_InsPayer_cpt_2025,
    on=['Billing_Physician','Plan_Name','InsPayer_Id','Procedure_Code'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge3_both_2025 = modi_merge3_2025[modi_merge3_2025['_merge'] == 'both']

ds_remaining2025_3 = modi_merge3_2025[modi_merge3_2025['_merge'] == 'left_only']
ds_remaining2025_3.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge3_2025.drop(columns=['_merge'], inplace=True)
modi_merge3_both_2025.drop(columns=['_merge'], inplace=True)

ds_remaining2025_3.drop(columns={'Count'}, inplace=True)

In [170]:
modi_merge3_both_2025['LAYER_INFO'] = 'amt_pro_plan_InsPayer_cpt_2025'

In [171]:
modi_merge3_both_2025.shape[0]

0

In [172]:
ds_remaining2025_3.shape[0]

453

## Step 14

In [173]:
modi_merge3_2025_n = ds_remaining2025_3.merge(
    amt_pro_plan_emc_cpt_2025,
    on=['Billing_Physician','Plan_Name','EMC_NO','Procedure_Code'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge3_both_2025_n = modi_merge3_2025_n[modi_merge3_2025_n['_merge'] == 'both']

ds_remaining2025_3_n = modi_merge3_2025_n[modi_merge3_2025_n['_merge'] == 'left_only']
ds_remaining2025_3_n.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge3_2025_n.drop(columns=['_merge'], inplace=True)
modi_merge3_both_2025_n.drop(columns=['_merge'], inplace=True)

ds_remaining2025_3_n.drop(columns={'Count'}, inplace=True)

In [174]:
modi_merge3_both_2025_n['LAYER_INFO'] = 'amt_pro_plan_emc_cpt_2025'

In [175]:
modi_merge3_both_2025_n.shape[0]

0

In [176]:
ds_remaining2025_3_n.shape[0]

453

## Step 15

In [179]:
modi_merge6 = ds_remaining2025_3_n.merge(
    amt_pro_plan_InsPayer_cpt,
    on=['Billing_Physician','Plan_Name','InsPayer_Id','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both4 = modi_merge6[modi_merge6['_merge'] == 'both']

modi_merge6.drop(columns=['_merge'], inplace=True)
modi_merge_both4.drop(columns=['_merge'], inplace=True)

In [180]:
modi_merge_both4['LAYER_INFO'] = 'amt_pro_plan_InsPayer_cpt'

In [181]:
modi_merge_both4.shape[0]

0

## Step 16

In [182]:
modi_merge7 = ds_remaining2025_3_n.merge(
    amt_pro_plan_emc_cpt,
    on=['Billing_Physician','Plan_Name','EMC_NO','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both5 = modi_merge7[modi_merge7['_merge'] == 'both']

modi_merge7.drop(columns=['_merge'], inplace=True)
modi_merge_both5.drop(columns=['_merge'], inplace=True)

In [183]:
modi_merge_both5['LAYER_INFO'] = 'amt_pro_plan_emc_cpt'

In [184]:
merged_both3 = merge_both_dfs(modi_merge_both4, modi_merge_both5)

In [185]:
merged_common_left5, merged_right_only3 = common_left_right_df(merged_both3)

In [186]:
merged_common_left5.shape[0], modi_merge_both5.shape[0]

(0, 0)

In [187]:
ds_remaining5 = ds_claim_remaining(ds_remaining2025_3_n, merged_common_left5)

In [188]:
ds_remaining5.shape[0]

453

## Step 17

In [189]:
# modi_merge8 = ds_remaining5.merge(
#     fee_schedule_ins_payer, 
#     on=['Billing_Physician', 'InsPayer_Id', 'Procedure_Code'], 
#     how='left', 
#     indicator=True
# )

# merged_common_left6 = modi_merge8[modi_merge8['_merge']=='both']

In [190]:
# merged_common_left6['LAYER_INFO'] = 'fee_schedule_ins_payer'

In [191]:
# ds_remaining6 = modi_merge8[modi_merge8['_merge'] == 'left_only']
# ds_remaining6.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [192]:
# ds_remaining6.shape[0], merged_common_left6.shape[0]

## Step 18

In [193]:
# modi_merge9 = ds_remaining6.merge(
#     fee_schedule_emc, 
#     on=['Billing_Physician', 'EMC_NO', 'Procedure_Code'], 
#     how='left', 
#     indicator=True
# )

# modi_merge9 = modi_merge9.drop_duplicates(subset={'Claim_No','Billing_Physician', 'EMC_NO', 'Process_Date','InsPayer_Id','Procedure_Code'})
# merged_common_left7 = modi_merge9[modi_merge9['_merge']=='both']

In [194]:
# merged_common_left7['LAYER_INFO'] = 'fee_schedule_emc'

In [195]:
# ds_remaining7 = modi_merge9[modi_merge9['_merge'] == 'left_only']
# ds_remaining7.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [196]:
# merged_common_left7.shape[0], ds_remaining7.shape[0]

In [197]:
#2025 Provider, InsPayer, Procedure Code

## Step 19

In [199]:
modi_merge9_2025 = ds_remaining5.merge(
    amt_pro_InsPayer_proc_2025,
    on=['Billing_Physician','InsPayer_Id','Procedure_Code'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge9_both_2025_n = modi_merge9_2025[modi_merge9_2025['_merge'] == 'both']

ds_remaining2025_9 = modi_merge9_2025[modi_merge9_2025['_merge'] == 'left_only']
ds_remaining2025_9.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge9_2025.drop(columns=['_merge'], inplace=True)
modi_merge9_both_2025_n.drop(columns=['_merge'], inplace=True)

ds_remaining2025_9.drop(columns={'Count'}, inplace=True)

In [200]:
modi_merge9_both_2025_n['LAYER_INFO'] = 'amt_pro_InsPayer_proc_2025'

In [201]:
modi_merge9_both_2025_n.shape[0]

163

In [202]:
ds_remaining2025_9.shape[0]

290

## Step 20

In [203]:
modi_merge9_2025_n = ds_remaining2025_9.merge(
    amt_pro_emc_proc_2025,
    on=['Billing_Physician','EMC_NO','Procedure_Code'],
    how='left',
    indicator=True
)
# # # Step 2: Filter based on _merge column
modi_merge9_both_2025_n1 = modi_merge9_2025_n[modi_merge9_2025_n['_merge'] == 'both']

ds_remaining2025_9_n = modi_merge9_2025_n[modi_merge9_2025_n['_merge'] == 'left_only']
ds_remaining2025_9_n.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge9_2025_n.drop(columns=['_merge'], inplace=True)
modi_merge9_both_2025_n1.drop(columns=['_merge'], inplace=True)

ds_remaining2025_9_n.drop(columns={'Count'}, inplace=True)

In [204]:
modi_merge9_both_2025_n1['LAYER_INFO'] = 'amt_pro_emc_proc_2025'

In [205]:
modi_merge9_both_2025_n1.shape[0]

26

In [206]:
ds_remaining2025_9_n.shape[0]

264

# Step 21

In [207]:
modi_merge10 = ds_remaining2025_9_n.merge(
    amt_pro_InsPayer_proc,
    on=['Billing_Physician','InsPayer_Id','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both6 = modi_merge10[modi_merge10['_merge'] == 'both']

ds_remaining7 = modi_merge10[modi_merge10['_merge'] == 'left_only']
ds_remaining7.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge10.drop(columns=['_merge'], inplace=True)
modi_merge_both6.drop(columns=['_merge'], inplace=True)

ds_remaining7.drop(columns={'Count'}, inplace=True)

In [208]:
modi_merge_both6['LAYER_INFO'] = 'amt_pro_InsPayer_proc'

In [209]:
modi_merge_both6.shape[0], ds_remaining7.shape[0]

(33, 231)

# Step 22

In [210]:
modi_merge11 = ds_remaining7.merge(
    amt_pro_emc_proc,
    on=['Billing_Physician','EMC_NO','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both7 = modi_merge11[modi_merge11['_merge'] == 'both']

ds_remaining2025_9 = modi_merge11[modi_merge11['_merge'] == 'left_only']
ds_remaining2025_9.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

modi_merge11.drop(columns=['_merge'], inplace=True)
modi_merge_both7.drop(columns=['_merge'], inplace=True)

ds_remaining2025_9.drop(columns={'Count'}, inplace=True)

In [211]:
modi_merge_both7['LAYER_INFO'] = 'amt_pro_emc_proc'

In [212]:
modi_merge_both7.shape[0], ds_remaining2025_9.shape[0]

(9, 222)

# Fee Schedule Up Down Testing

# Step 17

In [213]:
modi_merge8 = ds_remaining2025_9.merge(
    fee_schedule_ins_payer, 
    on=['Billing_Physician', 'InsPayer_Id', 'Procedure_Code'], 
    how='left', 
    indicator=True
)

merged_common_left6 = modi_merge8[modi_merge8['_merge']=='both']

In [214]:
merged_common_left6['LAYER_INFO'] = 'fee_schedule_ins_payer'

In [215]:
ds_remaining6 = modi_merge8[modi_merge8['_merge'] == 'left_only']
ds_remaining6.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [216]:
ds_remaining6.shape[0], merged_common_left6.shape[0]

(222, 0)

# Step 18

In [217]:
modi_merge9 = ds_remaining6.merge(
    fee_schedule_emc, 
    on=['Billing_Physician', 'EMC_NO', 'Procedure_Code'], 
    how='left', 
    indicator=True
)

modi_merge9 = modi_merge9.drop_duplicates(subset={'Claim_No','Billing_Physician', 'EMC_NO', 'Process_Date','InsPayer_Id','Procedure_Code'})
merged_common_left7 = modi_merge9[modi_merge9['_merge']=='both']

In [218]:
merged_common_left7['LAYER_INFO'] = 'fee_schedule_emc'

In [219]:
ds_remaining7_n = modi_merge9[modi_merge9['_merge'] == 'left_only']
ds_remaining7_n.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [220]:
merged_common_left7.shape[0], ds_remaining7_n.shape[0]

(0, 222)

# Fee Schedule Up Down Testing END

## Step 21

In [222]:
# modi_merge10 = ds_remaining7.merge(
#     amt_pro_InsPayer_proc,
#     on=['Billing_Physician','InsPayer_Id','Procedure_Code'],
#     how='left',
#     indicator=True
# )

# modi_merge_both6 = modi_merge10[modi_merge10['_merge'] == 'both']

# modi_merge10.drop(columns=['_merge'], inplace=True)
# modi_merge_both6.drop(columns=['_merge'], inplace=True)

In [223]:
# modi_merge_both6['LAYER_INFO'] = 'amt_pro_InsPayer_proc'

## Step 22 

In [224]:
# modi_merge11 = ds_remaining7.merge(
#     amt_pro_emc_proc,
#     on=['Billing_Physician','EMC_NO','Procedure_Code'],
#     how='left',
#     indicator=True
# )

# modi_merge_both7 = modi_merge11[modi_merge11['_merge'] == 'both']

# modi_merge11.drop(columns=['_merge'], inplace=True)
# modi_merge_both7.drop(columns=['_merge'], inplace=True)

In [225]:
# modi_merge_both7['LAYER_INFO'] = 'amt_pro_emc_proc'

In [226]:
# modi_merge_both6.shape[0], modi_merge_both7.shape[0]

In [227]:
# merged_both4 = merge_both_dfs(modi_merge_both6, modi_merge_both7)

In [228]:
# merged_common_left8, merged_right_only4 = common_left_right_df(merged_both4)

In [229]:
# ds_remaining8 = ds_claim_remaining(ds_remaining2025_9_n, merged_common_left8)

In [230]:
# merged_both4.shape[0], merged_common_left8.shape[0], ds_remaining8.shape[0]

In [231]:
merge12_n_2025 = ds_remaining7_n.merge(
    amt_pro_cpt_2025,
    on=['Billing_Physician','Procedure_Code'],
    how='left',
    indicator=True
)

merge_both8_2025 = merge12_n_2025[merge12_n_2025['_merge'] == 'both']
ds_remaining6_n_2025 = merge12_n_2025[merge12_n_2025['_merge'] == 'left_only']

merge12_n_2025.drop(columns=['_merge'], inplace=True)
merge_both8_2025.drop(columns=['_merge'], inplace=True)

In [232]:
merge_both8_2025['LAYER_INFO'] = 'amt_pro_cpt_2025'

In [233]:
ds_remaining6_n_2025.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [234]:
ds_remaining6_n_2025.drop(columns='Count', inplace=True)

In [235]:
ds_remaining6_n_2025.shape[0], merge_both8_2025.shape[0]

(115, 107)

## Step 23

In [238]:
modi_merge12 = ds_remaining6_n_2025.merge(
    amt_pro_cpt,
    on=['Billing_Physician','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both8 = modi_merge12[modi_merge12['_merge'] == 'both']

modi_merge12.drop(columns=['_merge'], inplace=True)
modi_merge_both8.drop(columns=['_merge'], inplace=True)
# modi_merge_both8.drop(columns=['Count'], inplace=True)

In [239]:
modi_merge_both8['LAYER_INFO'] = 'amt_pro_cpt'

In [241]:
# merged_both5 = merge_both_dfs(merged_right_only4, modi_merge_both8)

In [242]:
# merged_both5.shape[0],modi_merge_both8.shape[0],modi_merge12.shape[0]

In [243]:
# merged_common_left9, merged_right_only5 = common_left_right_df(merged_both5)

In [244]:
# merged_common_left9.shape[0], merged_right_only5.shape[0]

In [245]:
# ds_remaining9 = ds_claim_remaining(ds_remaining8, merged_common_left9)

In [246]:
# ds_remaining9.shape[0]

## Step 24

In [248]:
modi_merge13 = ds_remaining7_n.merge(
    amt_spec_InsPayer_state_cpt,
    on=['SPECIALIZATION_CODE','InsPayer_Id','PROVIDER_STATE','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both9 = modi_merge13[modi_merge13['_merge'] == 'both']

modi_merge13.drop(columns=['_merge'], inplace=True)
modi_merge_both9.drop(columns=['_merge'], inplace=True)

In [249]:
modi_merge_both9['LAYER_INFO'] = 'amt_spec_InsPayer_state_cpt'

In [250]:
modi_merge_both9.shape[0]

77

In [252]:
merged_both6 = merge_both_dfs(modi_merge_both8, modi_merge_both9)

In [253]:
# merged_both6['Count'].fillna(0, inplace=True)

In [254]:
# merged_both6

In [256]:
merged_common_left10, merged_right_only6 = common_left_right_df(merged_both6)

In [257]:
ds_remaining10 = ds_claim_remaining(ds_remaining7_n, merged_common_left10)

In [258]:
ds_remaining10.shape[0]

219

## Step 25

In [261]:
modi_merge14 = ds_remaining10.merge(
    amt_spec_emc_state_cpt,
    on=['SPECIALIZATION_CODE','EMC_NO','PROVIDER_STATE','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both10 = modi_merge14[modi_merge14['_merge'] == 'both']

modi_merge14.drop(columns=['_merge'], inplace=True)
modi_merge_both10.drop(columns=['_merge'], inplace=True)

In [262]:
modi_merge_both10['LAYER_INFO'] = 'amt_spec_emc_state_cpt'

In [263]:
merged_both7 = merge_both_dfs(merged_right_only6, modi_merge_both10)

In [264]:
merged_common_left11, merged_right_only7 = common_left_right_df(merged_both7)

In [265]:
ds_remaining11 = ds_claim_remaining(ds_remaining10, merged_common_left11)

In [266]:
ds_remaining11.shape[0]

142

## Step 26

In [267]:
modi_merge15 = ds_remaining11.merge(
    amt_spec_state_cpt,
    on=['SPECIALIZATION_CODE','PROVIDER_STATE','Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge_both11 = modi_merge15[modi_merge15['_merge'] == 'both']

modi_merge15.drop(columns=['_merge'], inplace=True)
modi_merge_both11.drop(columns=['_merge'], inplace=True)

In [268]:
modi_merge_both11['LAYER_INFO'] = 'amt_spec_state_cpt'

In [269]:
merged_both8 = merge_both_dfs(merged_right_only7, modi_merge_both11)

In [270]:
merged_common_left12, merged_right_only8 = common_left_right_df(merged_both8)

In [271]:
ds_remaining12 = ds_claim_remaining(ds_remaining11, merged_common_left12)

In [272]:
ds_remaining12.shape[0]

93

# Step 28

In [274]:
modi_merge_16_2025 = ds_remaining12.merge(
    amt_proc_2025, 
    on=['Procedure_Code'], 
    how='left', 
    indicator=True
)

modi_merge_16_2025_both = modi_merge_16_2025[modi_merge_16_2025['_merge']=='both']

In [275]:
modi_merge_16_2025_both['LAYER_INFO'] = 'amt_proc_2025'

In [276]:
ds_remaining_end_2025 = modi_merge_16_2025[modi_merge_16_2025['_merge'] == 'left_only']
ds_remaining_end_2025.drop(columns={'_merge','Avg_Approved_Amt'}, inplace=True)

In [277]:
ds_remaining_end_2025.drop(columns={'Count'}, inplace=True)

In [278]:
ds_remaining_end_2025.shape[0], modi_merge_16_2025_both.shape[0]

(0, 93)

## Step 27

In [280]:
modi_merge16 = ds_remaining_end_2025.merge(
    amt_proc,
    on=['Procedure_Code'],
    how='left',
    indicator=True
)

modi_merge16.drop(columns=['_merge'], inplace=True)

In [281]:
merged_common_left13 = modi_merge16

In [282]:
merged_common_left13['LAYER_INFO'] = 'amt_proc'

In [283]:
merged_common_left13['Avg_Approved_Amt'].fillna(merged_common_left13['Predictions'], inplace=True)

In [284]:
merged_common_left.shape[0]+merged_common_left1.shape[0]+merged_common_left2.shape[0]+merged_common_left3.shape[0]+merged_common_left4.shape[0]+merged_common_left5.shape[0]+merged_common_left6.shape[0]+merged_common_left7.shape[0]+merged_common_left10.shape[0]+merged_common_left11.shape[0]+merged_common_left12.shape[0]+merged_common_left13.shape[0]+modi_merge_both_2025.shape[0]+modi_merge_both_2025_n.shape[0]+ modi_merge_both_2025_n1.shape[0]+modi_merge_both_2025_n2.shape[0]+modi_merge3_both_2025.shape[0]+modi_merge3_both_2025_n.shape[0]+ modi_merge9_both_2025_n.shape[0]+modi_merge9_both_2025_n1.shape[0] +modi_merge_16_2025_both.shape[0]+merge_both8_2025.shape[0]

577

# Merging all Data Frames

In [287]:
final_df = pd.concat([medicare_modi_merge_both, medicare_modi_merge_inspayer_both,merged_common_left3,merged_common_left4,
modi_merge_both_2025,modi_merge_both_2025_n,modi_merge_both_2025_n1,modi_merge_both_2025_n2,merged_common_left,
merged_common_left1,merged_common_left2,modi_merge3_both_2025,modi_merge3_both_2025_n, merged_common_left5,                      
modi_merge9_both_2025_n,modi_merge9_both_2025_n1,modi_merge_both6,modi_merge_both7,merged_common_left6,
merged_common_left7,merged_common_left10, merged_common_left11,merged_common_left12,merged_common_left13, 
modi_merge_16_2025_both,merge_both8_2025]).reset_index(drop=True)

In [288]:
# final_df = pd.concat([medicare_modi_merge_both, medicare_modi_merge_inspayer_both, merged_common_left,merged_common_left1,merged_common_left2,merged_common_left3,merged_common_left4,merged_common_left5,merged_common_left8,merged_common_left9,merged_common_left10,merged_common_left11,merged_common_left12,merged_common_left13,modi_merge_both_2025,modi_merge_both_2025_n, modi_merge_both_2025_n1, modi_merge_both_2025_n2, modi_merge3_both_2025,modi_merge3_both_2025_n, modi_merge9_both_2025_n,modi_merge9_both_2025_n1]).reset_index(drop=True)

In [289]:
final_df.drop(columns={'Count','_merge'}, inplace=True)

In [290]:
final_df.shape

(619, 541)

In [291]:
final_df = final_df.drop_duplicates(subset={'Claim_No','Billing_Physician', 'EMC_NO', 'Process_Date','InsPayer_Id','Procedure_Code'})

In [292]:
final_df = final_df.drop(columns=[col for col in final_df.columns if col.endswith('_1') or col.endswith('_2')])

In [293]:
final_df['NewPredictions'] = final_df['Avg_Approved_Amt'] * final_df['units']

final_df.loc[final_df['Avg_Approved_Amt'] > final_df['Amount'], 'NewPredictions'] = \
    final_df['Amount'] * 0.44

In [295]:
final_df.shape

(512, 184)

In [296]:
final_df.shape[0],ds_claim.shape[0]

(512, 512)

In [297]:
ds_claim.Amount.sum(), final_df.Amount.sum(), ds_claim.Predictions.sum(), final_df.Predictions.sum()

(np.float64(31304.510000000002),
 np.float64(31304.51),
 np.float64(93877.17207931454),
 np.float64(93877.17207931452))

In [298]:
final_df.Avg_Approved_Amt.sum(), final_df.NewPredictions.sum()

(np.float64(16198.145600000002), np.float64(16207.445600000001))

In [299]:
# final_df[(final_df['Procedure_Code'] == '3078F') & (final_df['InsPayer_Id'] == 5349856)].sort_values(by='Avg_Approved_Amt').reset_index(drop=True)

In [300]:
# final_df['NewPredictions'] = final_df.apply(
#     lambda row: (
#         row['Avg_Approved_Amt'] * row['units']
#         if row['Avg_Approved_Amt'] != 0 and 
#            (abs(row['Predictions'] - row['Avg_Approved_Amt']) / row['Avg_Approved_Amt']) * 100 >= 1
#         else (
#             row['Predictions']
#             if row['Predictions'] < row['Amount']
#             else row['Amount'] * 0.44
#         )
#     ),
#     axis=1
# )

In [301]:
final_df.reset_index(drop=True, inplace=True)

In [302]:
# final_df.loc[final_df['NewPredictions']>final_df['Amount'], 'NewPredictions']=final_df['Amount']*0.44

In [303]:
final_df.Amount.sum(),final_df.Predictions.sum(), final_df.NewPredictions.sum()

(np.float64(31304.51),
 np.float64(93877.17207931452),
 np.float64(16207.445600000001))

In [304]:
final_df.loc[(final_df['Modi_Code1'] == '50') & (final_df['InsPayer_Id'] == 5002492), 'NewPredictions'] *= 1.5

In [305]:
final_df.NewPredictions.sum()

np.float64(16242.9256)

In [306]:
# final_df.head(20)

#  Updating Days

In [307]:
day_pro_EMC_stat_provstat.drop(columns={'Count'}, inplace=True)
day_pro_EMC_stat.drop(columns={'Count'}, inplace=True)
day_pro_EMC.drop(columns={'Count'}, inplace=True)
day_EMC.drop(columns={'Count'}, inplace=True)

day_pro_ins_stat_provstat.drop(columns={'Count'}, inplace=True)
day_pro_ins_stat.drop(columns={'Count'}, inplace=True)
day_pro_ins.drop(columns={'Count'}, inplace=True)
day_ins.drop(columns={'Count'}, inplace=True)

In [308]:
final_df.shape[0]

512

In [309]:
# final_df.head()

In [310]:
# final_df_days_greater_50 = final_df[final_df['Predicted_Days'] > 2].reset_index(drop=True)

In [311]:
# final_df_days_less_50 = final_df[final_df['Predicted_Days'] <= 2].reset_index(drop=True)

In [312]:
# final_df_days_less_50['Days'] = final_df_days_less_50['Predicted_Days']

In [313]:
final_df_days_greater_50 = deepcopy(final_df)

In [314]:
final_df_days_greater_50.shape[0]

512

In [316]:
day_pro_ins_stat['Billing_Physician']=day_pro_ins_stat['Billing_Physician'].astype(int)
day_pro_ins_stat['InsPayer_Id']=day_pro_ins_stat['InsPayer_Id'].astype(int)
day_pro_ins['Billing_Physician'] = day_pro_ins['Billing_Physician'].astype(int)
day_pro_ins['InsPayer_Id'] = day_pro_ins['InsPayer_Id'].astype(int)
day_ins['InsPayer_Id'] = day_ins['InsPayer_Id'].astype(int)

In [317]:
day_pro_EMC_stat['Billing_Physician']=day_pro_ins_stat['Billing_Physician'].astype(int)
day_pro_EMC['Billing_Physician'] = day_pro_ins['Billing_Physician'].astype(int)

In [319]:
final_df_days_greater_50['PROVIDER_STATE'] = final_df_days_greater_50['PROVIDER_STATE'].astype(str).str.strip().str.upper()
day_pro_EMC_stat_provstat['PROVIDER_STATE'] = day_pro_EMC_stat_provstat['PROVIDER_STATE'].astype(str).str.strip().str.upper()
day_pro_ins_stat_provstat['PROVIDER_STATE'] = day_pro_ins_stat_provstat['PROVIDER_STATE'].astype(str).str.strip().str.upper()

In [320]:
days_merge = final_df_days_greater_50.merge(day_pro_ins_stat_provstat, on=['Billing_Physician','InsPayer_Id','Insurance_State','PROVIDER_STATE']
                               ,how='left', indicator=True)
days_prov_InsPayer_state_prov_state_df = days_merge[days_merge['_merge'] == 'both']

In [321]:
days_prov_InsPayer_state_prov_state_df.shape[0]

363

In [322]:
days_remaining = days_merge[days_merge['_merge'] == 'left_only']
days_remaining.drop(columns={'_merge','Days'}, inplace=True)

In [323]:
days_remaining.shape[0]

149

In [324]:
days_merge1 = days_remaining.merge(day_pro_EMC_stat_provstat, on=['Billing_Physician','EMC_NO','Insurance_State','PROVIDER_STATE']
                               , how='left', indicator=True)
days_prov_EMC_state_prov_state_df = days_merge1[days_merge1['_merge'] == 'both']

In [325]:
days_prov_EMC_state_prov_state_df.shape[0]

58

In [326]:
# days_prov_EMC_state_prov_state_df.head()

In [327]:
days_remaining1 = days_merge1[days_merge1['_merge'] == 'left_only']
days_remaining1.drop(columns={'_merge','Days'}, inplace=True)

In [328]:
days_remaining1.shape[0]

91

In [329]:
days_merge2 = days_remaining1.merge(day_pro_ins_stat, on=['Billing_Physician','InsPayer_Id','Insurance_State']
                               , how='left', indicator=True)
days_prov_InsPayer_state_df = days_merge2[days_merge2['_merge'] == 'both']

In [330]:
days_prov_InsPayer_state_df.shape[0]

0

In [331]:
days_remaining2 = days_merge2[days_merge2['_merge'] == 'left_only']
days_remaining2.drop(columns={'_merge','Days'}, inplace=True)

In [332]:
days_remaining2.shape[0]

91

In [333]:
days_merge3 = days_remaining2.merge(day_pro_EMC_stat, on=['Billing_Physician','EMC_NO','Insurance_State']
                               ,how='left', indicator=True)
days_merge3 = days_merge3.drop_duplicates(subset={'Claim_No','Billing_Physician', 'EMC_NO', 'Process_Date','InsPayer_Id','Procedure_Code'})
days_prov_emc_state_df = days_merge3[days_merge3['_merge'] == 'both']

In [334]:
days_prov_emc_state_df.shape[0]

0

In [335]:
days_remaining3 = days_merge3[days_merge3['_merge'] == 'left_only']
days_remaining3.drop(columns={'_merge','Days'}, inplace=True)

In [336]:
days_merge4 = days_remaining3.merge(day_pro_ins, on=['Billing_Physician','InsPayer_Id']
                               ,how='left', indicator=True)
days_prov_InsPayer_df = days_merge4[days_merge4['_merge'] == 'both']

In [337]:
days_prov_InsPayer_df.shape[0]

4

In [338]:
days_remaining4 = days_merge4[days_merge4['_merge'] == 'left_only']
days_remaining4.drop(columns={'_merge','Days'}, inplace=True)

In [339]:
days_merge5 = days_remaining4.merge(day_pro_EMC, on=['Billing_Physician','EMC_NO']
                               , how='left', indicator=True)
days_prov_emc_df = days_merge5[days_merge5['_merge'] == 'both']

In [340]:
days_prov_emc_df.shape[0]

9

In [341]:
days_remaining5 = days_merge5[days_merge5['_merge'] == 'left_only']
days_remaining5.drop(columns={'_merge','Days'}, inplace=True)

In [342]:
days_remaining5.shape[0]

78

In [343]:
days_merge6 = days_remaining5.merge(day_ins, on=['InsPayer_Id']
                               , how='left', indicator=True)
days_ins_df = days_merge6[days_merge6['_merge'] == 'both']

In [344]:
days_ins_df.shape[0]

0

In [345]:
days_remaining6 = days_merge6[days_merge6['_merge'] == 'left_only']
days_remaining6.drop(columns={'_merge','Days'}, inplace=True)

In [346]:
days_remaining6.shape[0]

78

In [347]:
days_merge7 = days_remaining6.merge(day_EMC, on=['EMC_NO']
                               , how='left', indicator=True)
days_emc_df = days_merge7

In [348]:
days_emc_df.shape[0]

78

In [350]:
days_prov_InsPayer_state_prov_state_df.drop(columns={'_merge'}, inplace=True)
days_prov_EMC_state_prov_state_df.drop(columns={'_merge'}, inplace=True)
days_prov_emc_state_df.drop(columns={'_merge'}, inplace=True)
days_prov_InsPayer_state_df.drop(columns={'_merge'}, inplace=True)
days_prov_emc_df.drop(columns={'_merge'}, inplace=True)
days_prov_InsPayer_df.drop(columns={'_merge'}, inplace=True)
days_emc_df.drop(columns={'_merge'}, inplace=True)
days_ins_df.drop(columns={'_merge'}, inplace=True)

In [351]:
days_final_df = pd.concat([days_prov_InsPayer_state_prov_state_df,days_prov_EMC_state_prov_state_df,days_prov_emc_state_df,days_prov_InsPayer_state_df,days_prov_emc_df,days_prov_InsPayer_df,days_emc_df,days_ins_df])
# final_days_amt  = pd.concat([days_final_df,final_df_days_less_50])

In [352]:
final_days_amt  = deepcopy(days_final_df)

In [353]:
# final_days_amt.head()

In [354]:
final_days_amt[final_days_amt['Days'].isna()].shape[0]

78

In [355]:
final_days_amt.Days.unique()

array([15.,  6.,  8.,  2.,  7.,  3., 23., 35.,  5., 28., 21., 11., 14.,
       36.,  9., 18., 48., nan])

In [356]:
final_days_amt['Days'].fillna(final_days_amt['Predicted_Days'], inplace=True)
final_days_amt['Days'] = final_days_amt['Days'].astype(int)
final_days_amt['Predicted_Days'] = final_days_amt['Predicted_Days'].astype(int)

In [357]:
final_days_amt['NewDays'] = final_days_amt.apply(
    lambda row: row['Days'] if abs(row['Days'] - row['Predicted_Days']) > 5 else row['Predicted_Days'],
    axis=1
)

In [358]:
# final_days_amt.tail(2)

In [359]:
final_days_amt[final_days_amt['NewDays']>30]['Claim_No'].nunique()

22

In [360]:
final_days_amt.loc[final_days_amt['NewDays'] > 30, 'NewDays'] = 14
final_days_amt.loc[final_days_amt['NewDays'] < 1, 'NewDays'] = 14
# final_days_amt['Days'].fillna(30, inplace=True)
# final_days_amt['Days'] = final_days_amt['Days'].astype(int)

In [361]:
# Ensure 'Predicted_Date' is in datetime format
final_days_amt['Process_Date'] = pd.to_datetime(final_days_amt['Process_Date'], errors='coerce')

# Subtract timedelta
final_days_amt['NewPredictedDate'] = final_days_amt['Process_Date'] + pd.to_timedelta(final_days_amt['NewDays'], unit='D')
final_days_amt['NewPredictedDate'].fillna(final_days_amt['Predicted_Date'], inplace=True)
final_days_amt['NewDays'].fillna(final_days_amt['Predicted_Days'], inplace=True)

In [362]:
# Verify the distribution
final_days_amt_distribution = final_days_amt["NewPredictedDate"].dt.day_name().value_counts(normalize=True)
print(final_days_amt_distribution)

NewPredictedDate
Thursday     0.353516
Friday       0.242188
Wednesday    0.171875
Saturday     0.072266
Monday       0.064453
Sunday       0.050781
Tuesday      0.044922
Name: proportion, dtype: float64


# Sunday Adjustment

In [363]:
final_all_data = deepcopy(final_days_amt)

In [364]:
final_all_data['Predicted_Date'] = final_all_data['NewPredictedDate']
final_all_data['Predictions'] = final_all_data['NewPredictions']
final_all_data['Predicted_Days'] = final_all_data['NewDays']

In [365]:
final_all_data.drop(columns={'Avg_Approved_Amt','NewPredictions','Days','NewDays','NewPredictedDate'}, inplace=True)

In [366]:
final_all_data['Predicted_Date'] = pd.to_datetime(final_all_data['Predicted_Date'])

In [367]:
final_sunday = final_all_data[final_all_data['Predicted_Date'].dt.weekday==6]

In [368]:
final_non_sunday = final_all_data[final_all_data['Predicted_Date'].dt.weekday != 6]

In [369]:
# Weekdays percentages including Sunday
weekdays_percentages = {
    'Monday': 9.256319,
    'Tuesday': 16.552252,
    'Wednesday': 19.465113,
    'Thursday': 18.814590,
    'Friday': 17.633205,
    'Saturday': 15.531761,
    'Sunday': 2.746760
}

# Normalize percentages
total_percentage = sum(weekdays_percentages.values())
normalized_percentages = {day: percent / total_percentage for day, percent in weekdays_percentages.items()}

# Add a column for day of the week
final_sunday["Day_of_Week"] = final_sunday["Predicted_Date"].dt.day_name()

# Filter rows with Sunday dates
sundays_df = final_sunday[final_sunday["Day_of_Week"] == "Sunday"]

# Group by Practice_Code and Predicted_Date, count unique Claim_No
grouped = sundays_df.groupby(["Practice_Code", "Predicted_Date", "Procedure_Code"])["Claim_No"].nunique()
valid_practice_codes = grouped[grouped > 7].index

# Redistribute rows for valid Practice_Code and Predicted_Date combinations
redistributed_rows = []
claim_date_mapping = {}  # Store the mapping of Claim_No to NewDate

for (practice_code, sunday_date, procedure_code), group in sundays_df.groupby(["Practice_Code", "Predicted_Date", "Procedure_Code"]):
    if (practice_code, sunday_date, procedure_code) in valid_practice_codes:
        unique_claims = group["Claim_No"].drop_duplicates().reset_index(drop=True)
        total_claims = len(unique_claims)

        # Calculate how many rows to allocate to each weekday based on percentages
        allocation = {day: int(total_claims * normalized_percentages[day]) for day in weekdays_percentages}
        remaining = total_claims - sum(allocation.values())

        # Distribute any remaining rows evenly
        weekdays = list(weekdays_percentages.keys())
        for i in range(remaining):
            allocation[weekdays[i % len(weekdays)]] += 1

        # Generate a mapping of weekdays to dates (relative to Sunday)
        weekday_dates = {
            'Monday': sunday_date + timedelta(days=1),
            'Tuesday': sunday_date + timedelta(days=2),
            'Wednesday': sunday_date + timedelta(days=3),
            'Thursday': sunday_date - timedelta(days=3),
            'Friday': sunday_date - timedelta(days=2),
            'Saturday': sunday_date - timedelta(days=1),
            'Sunday': sunday_date
        }

        # Assign dates to `Claim_No` proportionally
        assigned_dates = []
        for day, count in allocation.items():
            assigned_dates.extend([weekday_dates[day]] * count)

        # Shuffle dates to match the order of unique Claim_No
        assigned_dates = assigned_dates[:len(unique_claims)]

        # Map unique Claim_No to new dates (ensure same date for each Claim_No)
        for claim, new_date in zip(unique_claims, assigned_dates):
            claim_date_mapping[claim] = new_date

        # Now assign the same NewDate for all rows with the same Claim_No
        for _, row in group.iterrows():
            redistributed_rows.append({
                "Practice_Code": row["Practice_Code"],
                "Claim_No": row["Claim_No"],
                "Procedure_Code": row["Procedure_Code"],
                "Predictions": row["Predictions"],
                "Original_Date": row["Predicted_Date"],
                "NewDate": claim_date_mapping[row["Claim_No"]],  # Ensure same NewDate for all rows with the same Claim_No
            })
    else:
        # If the condition is not met, just add the original rows back to redistributed_rows
        for _, row in group.iterrows():
            redistributed_rows.append({
                "Practice_Code": row["Practice_Code"],
                "Claim_No": row["Claim_No"],
                "Procedure_Code": row["Procedure_Code"],
                "Predictions": row["Predictions"],
                "Original_Date": row["Predicted_Date"],
                "NewDate": row["Predicted_Date"],  # No change in date
            })

# Create a new dataframe with redistributed rows
redistributed_df = pd.DataFrame(redistributed_rows)

In [370]:
# redistributed_df.head(2)

In [371]:
redistributed_df["Claim_No"] = pd.to_numeric(redistributed_df["Claim_No"], errors="coerce")
new_redistributed_df = redistributed_df.sort_values(by=['Practice_Code','Claim_No'])
new_fin = new_redistributed_df.drop_duplicates(subset=['Practice_Code','Claim_No','Procedure_Code'])
new_fin.drop(columns={'Predictions','Original_Date'}, inplace=True)
neww_sunday = final_sunday.merge(new_fin, how='left', on=['Practice_Code','Claim_No','Procedure_Code'])
neww_sunday['NewDays'] = (neww_sunday['NewDate']-neww_sunday['Predicted_Date']).dt.days
neww_sunday['NewPredicted_Days'] = neww_sunday['Predicted_Days']+neww_sunday['NewDays']
# Convert 'Predicted_Date' to datetime
neww_sunday['Predicted_Date'] = pd.to_datetime(neww_sunday['Predicted_Date'])

# Update 'Predicted_Date' where 'NewPredicted_Days' <= 0
neww_sunday.loc[neww_sunday['NewPredicted_Days'] <= 0, 'Predicted_Date'] = (
    neww_sunday['Predicted_Date'] + pd.to_timedelta(30, unit='days')
)

neww_sunday.loc[neww_sunday['NewPredicted_Days'] <= 0, 'Predicted_Days'] = 30
neww_sunday['Predicted_Days'] = neww_sunday['NewPredicted_Days']
neww_sunday['Predicted_Date'] = neww_sunday['Predicted_Date'] + pd.to_timedelta((neww_sunday['NewDate'] - neww_sunday['Predicted_Date']).dt.days, unit='D')

neww_sunday.drop(columns={'Day_of_Week','NewDate','NewDays','NewPredicted_Days'}, inplace=True)

final_prepared_data = pd.concat([final_non_sunday, neww_sunday])
duplicates = final_prepared_data[final_prepared_data.duplicated(subset=['Claim_No', 'Billing_Physician', 'Process_Date', 'InsPayer_Id', 'Procedure_Code','Modi_Code1','Modi_Code2','Modi_Code3'])]

final_prepared_data.drop_duplicates(subset=['Claim_No', 'Billing_Physician', 'Process_Date', 'InsPayer_Id', 'Procedure_Code','Modi_Code1','Modi_Code2','Modi_Code3'], inplace=True)


### If there will be no Sunday in predicted date then ignore above two cells and uncoomnent this one

In [372]:

# final_prepared_data = final_all_data.copy()

## Continue 

In [373]:
# final_prepared_data.head(2)

In [374]:
# Calculate the minimum Predicted_Date for each group"
final_prepared_data['Predicted_Date'] = final_prepared_data.groupby(['Claim_No', 'Process_Date', 'InsPayer_Id'])['Predicted_Date'].transform('min')

In [375]:
final_prepared_data['Predicted_Days'] = final_prepared_data.groupby(['Claim_No', 'Process_Date', 'InsPayer_Id'])['Predicted_Days'].transform('min')

In [376]:
# Verify the distribution
final_prepared_data_distribution = final_prepared_data["Predicted_Date"].dt.day_name().value_counts(normalize=True)
print(final_prepared_data_distribution)

Predicted_Date
Thursday     0.353516
Friday       0.242188
Wednesday    0.171875
Saturday     0.072266
Monday       0.064453
Sunday       0.050781
Tuesday      0.044922
Name: proportion, dtype: float64


In [377]:
all_finalized = final_prepared_data[['Practice_Code','Att_Physician','Claim_No','DOS','Process_Date','InsPayer_Id','InsPayer_Description','Insurance_State','Procedure_Code','units','Predicted_Date','Predicted_Days','Amount','Predictions']]

In [378]:
all_finalized.rename(columns={'Predicted_Date':'Expected_Date','Predicted_Days':'Expected_Days',
                              'Amount':'Charge_Amount','Predictions':'Expected_Amount','units':'Units'}, inplace=True)

In [379]:
da = pd.to_datetime(all_finalized['Process_Date'].unique()[0]).date()

In [380]:
final_prepared_data_new = final_prepared_data[['Practice_Code','Prac_Name','Prac_State','Billing_Percentage',
'Claim_No','Amt_Due','DOS','BILL_DATE','Pri_Status','Sec_Status','Oth_Status','Pat_Status','DOE','Process_Date',
'Patient_Account','Patient_First_Name','Patient_Last_Name','PATIENT_GENDER','Relationship','Att_Physician','PROVIDER_NAME',
'Billing_Physician','InsPayer_Id','EMC_NO','InsName_Id','Insurance_Id','InsPayer_Description','Plan_Name','Insurance_State',
'Claim_Insurance_Id','Policy_Number','SUBMISSION_STATUS','Procedure_Code','Sequence_No','POS','DX_Code1','DX_Code2',
'DX_Code3','DX_Code4','DX_Code5','DX_Code6','DX_Code7','DX_Code8','Modi_Code1','Modi_Code2','Modi_Code3',
'Modi_Code4','DX_Pointer1','DX_Pointer2','DX_Pointer3','DX_Pointer4','Is_Self_Pay','Is_Active',
'SPECIALIZATION_CODE','units','Claim_Total','Amount','Predicted_Date','Predicted_Days','Predictions','Location_Code','LAYER_INFO']]

In [381]:
final_prepared_data_new.columns = final_prepared_data_new.columns.str.upper()

In [382]:
final_prepared_data_new.rename(columns={'RELATIONSHIP':'PAT_RELATIONSHIP','PREDICTED_DATE':'EXPECTED_PAYMENT_DATE','PREDICTED_DAYS':'EXPECTED_PAYMENT_DAYS','PREDICTIONS':'EXPECTED_APPROVED_AMT'}, inplace=True)

In [383]:
final_prepared_data_new.EXPECTED_APPROVED_AMT.sum()

np.float64(16242.925600000002)

In [384]:
# # In 2025 Medicare processe 1.6% of approved amount as CO-253, sop applying that on Medicare claims
# final_prepared_data_new.loc[final_prepared_data_new['INSPAYER_ID'] == 5002492,'EXPECTED_APPROVED_AMT'] = final_prepared_data_new.loc[final_prepared_data_new['INSPAYER_ID'] == 5002492, 'EXPECTED_APPROVED_AMT'] * (1 - 0.016)

In [385]:
# In 2025 Medicare processe 1.6% of approved amount as CO-253, so for 2025 Fee Schedule we don't need to deduct this amount
mask = (final_prepared_data_new['INSPAYER_ID'] == 5002492) &  \
       (~final_prepared_data_new['LAYER_INFO'].astype(str).str.contains('2025'))

final_prepared_data_new.loc[mask, 'EXPECTED_APPROVED_AMT'] *= (1 - 0.016)

In [386]:
final_prepared_data_new.EXPECTED_APPROVED_AMT.sum()

np.float64(16184.075129600002)

In [387]:
# final_prepared_data_new[(final_prepared_data_new['INSPAYER_ID']==5002492) & (final_prepared_data_new['PROCEDURE_CODE']=='99213')]

In [388]:
# final_prepared_data_new[(final_prepared_data_new['INSPAYER_ID']==5002492) & (final_prepared_data_new['PROCEDURE_CODE']=='99213')]

In [389]:
final_prepared_data_new['EXPECTED_PAID_AMT'] = 0

In [391]:
final_prepared_data_new['EXPECTED_APPROVED_AMT'] = final_prepared_data_new['EXPECTED_APPROVED_AMT'].round(2)
final_prepared_data_new['EXPECTED_PAID_AMT'] = final_prepared_data_new['EXPECTED_PAID_AMT'].round(2)

In [392]:
final_prepared_data_new.AMOUNT.sum(), final_prepared_data_new.EXPECTED_APPROVED_AMT.sum()

(np.float64(31304.510000000002), np.float64(16184.08))

## Provider Payer 

In [393]:
pro_pay = provider_payer()

In [394]:
pro_pay.rename(columns={'Provider_Code':'BILLING_PHYSICIAN','InsPayer_Id':'INSPAYER_ID'}, inplace=True)
final_prepared_provider_payer = final_prepared_data_new.merge(pro_pay, on=['BILLING_PHYSICIAN','INSPAYER_ID'], how='inner').reset_index(drop=True)
du_df = final_prepared_provider_payer[final_prepared_provider_payer.duplicated(subset=['CLAIM_NO', 'PROCEDURE_CODE'], keep=False)]
final_prepared_provider_payer.drop_duplicates(subset=['CLAIM_NO','PROCEDURE_CODE'], inplace=True)

In [395]:
final_prepared_provider_payer.EXPECTED_APPROVED_AMT.sum(),final_prepared_provider_payer.shape[0]

(np.float64(16184.08), 512)

In [396]:
missing_claims = set(main_df['Claim_No'].unique()) - set(final_prepared_provider_payer['CLAIM_NO'].unique())
missing_claims_list = list(missing_claims)
missing_claims_list = [int(x) for x in missing_claims_list]

# Non SDF Claims

In [397]:
# excluded_claims

In [398]:
resubmitted = all_submitted[~all_submitted['PRI_STATUS'].isin(['N','B'])].reset_index(drop=True)
paper_submission = all_submitted[all_submitted['SUBMISSION_TYPE'].isin(['QRD', 'DME'])].reset_index(drop=True)
wrong_insurance = all_submitted[all_submitted['CLAIM_NO'].isin(excluded_claims)].reset_index(drop=True)
out_network_provider_payer = all_submitted[all_submitted['CLAIM_NO'].isin(missing_claims_list)].reset_index(drop=True)

In [399]:
resubmitted.shape[0]

20

In [400]:
# List to store the DataFrames that need to be concatenated
dfs_to_concat = []

# Add 'NON_SDF_REASON' column to each DataFrame if not empty and then append to the list
if resubmitted.shape[0] > 0:
    resubmitted['NON_SDF_REASON'] = 'Resubmitted'
    dfs_to_concat.append(resubmitted)

if paper_submission.shape[0] > 0:
    paper_submission['NON_SDF_REASON'] = 'Paper Submission'
    dfs_to_concat.append(paper_submission)

if wrong_insurance.shape[0] > 0:
    wrong_insurance['NON_SDF_REASON'] = 'Submitted to Incorrect Insurance'
    dfs_to_concat.append(wrong_insurance)

if out_network_provider_payer.shape[0] > 0:
    out_network_provider_payer['NON_SDF_REASON'] = 'Provider Payer Not In Network'
    dfs_to_concat.append(out_network_provider_payer)

# Concatenate all DataFrames if there are any in the list
if dfs_to_concat:
    non_sdf_final = pd.concat(dfs_to_concat, ignore_index=True)
else:
    non_sdf_final = pd.DataFrame()  # Return an empty DataFrame if no DataFrames were appended


In [401]:
non_sdf_final = non_sdf_final.drop_duplicates(subset='CLAIM_NO')

In [402]:
non_sdf_final.CLAIM_NO.nunique()

21

In [403]:
# non_sdf_final

In [404]:
def non_sdf_data_dump(df):
    try:
        engine = db.create_engine('mssql+pymssql://user:password@ip/db')
    except Exception as e:
        print(e,'Unable to connect to DB')

    try:
        with engine.begin() as conn:
            df.to_sql('table', conn, if_exists='append', index=False)
            print('Total Data Dumped..:', df.shape[0])
    except Exception as e:
        print('Error in data Insertion:', e)

In [405]:
#Dumping non sdf data to DB
if non_sdf_final.shape[0]>0:
    non_sdf_data_dump(non_sdf_final)
else:
    print("No Non SDF Claims")

Total Data Dumped..: 21


In [406]:
final_prepared_provider_payer.shape[0]

512

In [407]:
final_prepared_provider_payer.CLAIM_NO.nunique()

148

In [408]:
final_prepared_provider_payer[final_prepared_provider_payer['PRACTICE_CODE']==111].shape[0]

54

In [409]:
final_prepared_provider_payer.PRACTICE_CODE.unique()

array([100, 111])

In [410]:
final_prepared_provider_payer.AMOUNT.sum(), final_prepared_provider_payer.EXPECTED_APPROVED_AMT.sum()

(np.float64(31304.510000000002), np.float64(16184.08))

In [411]:
final_prepared_provider_payer.PRACTICE_CODE.unique()

array([100, 111])

# Dump Data

In [413]:
# ddf = pd.read_csv("pred_fin_new.csv")

In [414]:
def data_dump(df):
    try:
        engine = db.create_engine('mssql+pymssql://user:password@ip/db')
    except Exception as e:
        print(e,'Unable to connect to DB')

    try:
        with engine.begin() as conn:
            df.to_sql('table', conn, if_exists='append', index=False)
            print('Total Data Dumped..:', df.shape[0])
    except Exception as e:
        print('Error in data Insertion:', e)

In [415]:
# data_dump(ddf)

In [416]:
# final_prepared_data_new.to_csv("dr_haq_sep_to_jan_pred.csv", index=False)

# Dump Data Testt

In [417]:
def data_dump_test(df):
    try:
        engine = db.create_engine('mssql+pymssql://user:password@ip/db')
    except Exception as e:
        print(e,'Unable to connect to DB')

    try:
        with engine.begin() as conn:
            df.to_sql('table', conn, if_exists='replace', index=False)
            print('Total Data Dumped..:', df.shape[0])
    except Exception as e:
        print('Error in data Insertion:', e)

In [418]:
data_dump_test(final_prepared_provider_payer)

Total Data Dumped..: 512


# END

# Email Part

In [ ]:
def Generate_Mail(excel_filename):
    print('Sending Email...')
    FROM = ""
    TO=['']
    CC=['']
    SUBJECT=""
    server=''
    html_template=''
    html_template1= f"""
    <html>
    <head></head>
    <body>

        <p>Hi,<br>
    <br>
    Please have attached forecast for the date of: {da}:

        <p>
        THIS IS A SYSTEM GENERATED MAIL BY DS TEAM.
    <p>
        
      </body>
    </html>
    """
    msg=MIMEMultipart()
    part = MIMEBase('application', 'octet-stream')
    msg['From'] = FROM
    msg['To'] = COMMASPACE.join(TO)
    msg['Cc'] = COMMASPACE.join(CC)
    toAddress = TO+CC
    msg['Date'] = formatdate(localtime=True)
    msg['Subject'] = SUBJECT
    msg.add_header('Content-Type','text/html')
    msg.attach(MIMEText(html_template1, 'html'))
    # Attach Excel file to the email
    with open(excel_filename, 'rb') as attachment:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(attachment.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename= {excel_filename}',)

    msg.attach(part)
    
    smtp = smtplib.SMTP(server, port = 587)
    smtp.starttls()
    smtp.login('', '')
    smtp.sendmail(FROM, toAddress, msg.as_string())
    smtp.close()
    print('Email has been sent.')
    
    return 0


In [ ]:
def main():
    print("Service Started..", datetime.datetime.now())
    #da = pd.to_datetime(final_prepared_data_new['Process_Date'].unique()[0]).date()
    excel_filename = 'SameDayFund.xlsx'
    final_prepared_data_new.to_excel(excel_filename, index=False)
    if len(final_prepared_data_new)>0:
        Generate_Mail(excel_filename)
    else:
        print("Error in Sending Email..")

In [ ]:
main()